# Fairness Transfer across States and Years with Bayesian Networks
**ICT 5143 · Algorithms, Fairness and Graphical Models Laboratory · Capstone Problem 5**

> *Can updating a small number (≤ 2) of shifted graphical-model factors preserve equalized odds more efficiently than
> global adaptation when only a small labelled target sample (n = 200 or 1,000) is available?*

This notebook is the whole project: it downloads ACS PUMS person files for California, Texas and New York (2018,
2019) from the U.S. Census Bureau, builds the Folktables ACSIncome benchmark, learns a constrained Bayesian network per
domain, verifies exact inference, and runs every mandatory experiment, ablation, robustness test and statistical test
of Problem 5. Part 1 defines the code; Part 2 runs it.

**Google Colab:** `Runtime → Run all` on a CPU runtime. Nothing has to be uploaded. At the end the results folder is
zipped and downloaded (`p5_results.zip`).

| `RUN_MODE` (next cell) | Use | Time (estimate) |
|---|---|---|
| `"full"` (default) | the reported numbers | about 13 min on Colab (CPU), about 30 min on a 4-core laptop |
| `"quick"` | fewer budgets, seeds and replicates; a check only, **never report** | well under half of that |

Outputs: `results/metrics` (per-run, per-group and paired results), `results/tables`, `results/figures`,
`results/logs`, `results/results_summary.md`, `results/reproducibility.json`, `results/manifest.csv`.

In [ ]:
import os, sys, json, time, zipfile, hashlib, platform, warnings, itertools, functools, subprocess, importlib
import urllib.request
from collections import Counter
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:   # --no-deps: never upgrade Colab's numpy / pandas / scikit-learn; the three packages are cross-checks only
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "folktables==0.0.12", "pgmpy",
                    "fairlearn", "scikit-base", "opt_einsum", "narwhals"], check=False)

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import gammaln, logsumexp, expit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
try:
    from IPython.display import display, Markdown
except ImportError:                                     # plain Python
    display, Markdown = print, str
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## Part 1 · Code

### 1.1 Settings (frozen before the reported run) and output helpers
Every function reads `CFG` when it runs, so changing a value here (or `CFG["delta_ba"] = 0.01` in a later cell) acts
everywhere.

In [ ]:
RUN_MODE = os.environ.get("P5_RUN_MODE", "full")      # "full" (reported) or "quick" (check only, never report)
assert RUN_MODE in ("full", "quick")
BASE_DIR = Path(os.environ.get("P5_BASE_DIR", "/content" if IN_COLAB else os.getcwd()))

CFG = {
    "run_mode": RUN_MODE, "config_file": "problem5_fairness_transfer_bn.ipynb",
    # data: Folktables ACSIncome; Y = 1 iff PINCP > 50,000 nominal dollars; household-level splits
    "data_source": "acs_pums", "states": ["CA", "TX", "NY"], "years": [2018, 2019], "income_threshold": 50000,
    "split_fracs": [0.6, 0.2, 0.2], "split_seed": 5143,
    "pairs": [["CA18", "TX18"], ["CA18", "NY18"], ["TX18", "CA18"], ["TX18", "NY18"], ["NY18", "CA18"],
              ["NY18", "TX18"], ["CA18", "CA19"], ["TX18", "TX19"], ["NY18", "NY19"]],
    # graph: tiers, required R,X edges, in-degree cap, held-out structure score, smoothing, outcome CPD, inference check
    "tiers": {"R": 0, "X": 0, "C": 1, "E": 2, "W": 3, "O": 3, "H": 4, "Y": 5},
    "within_tier_allowed": [["W", "O"], ["O", "W"]],
    "required_edges": [["R", "X"], ["R", "C"], ["R", "E"], ["R", "W"], ["R", "O"], ["R", "H"], ["X", "C"], ["X", "E"],
                       ["X", "W"], ["X", "O"], ["X", "H"]],
    "max_indegree": 4, "structure_score": "heldout", "alpha0": 2.0, "alpha_race": 10.0, "outcome_l2_C": 1.0,
    "inference_check_domain": "CA18", "inference_queries": 20, "inference_tol": 1.0e-9,
    # fairness: allowed balanced-accuracy loss; group support for the primary (100) and secondary (30) gaps
    "delta_ba": 0.02, "min_eval": 100, "min_eval_secondary": 30, "min_fit": 20,
    # BN-LFA (proposed)
    "max_factors": 2, "tilt_sigma": 1.0, "alpha_grid": [1, 3, 10, 30, 100, 300, 1000, 3000, 10000], "glm_sigma": 0.5,
    "lfa_weight_cap_quantile": 0.995,
    # baselines
    "iw_source_rows": 30000, "iw_domain_C": 0.1, "iw_weight_cap_quantile": 0.99, "retrain_C_grid": [0.1, 1.0, 10.0],
    "cv_folds": 3, "gbt_t_params": {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "subsample": 0.8},
    "hgb_params": {"max_iter": 200, "learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20,
                   "l2_regularization": 1.0, "early_stopping": False},
    # budgets and seeds
    "budgets": [0, 100, 200, 500, 1000, 2000], "confirmatory_budgets": [200, 1000], "seeds": [0, 1, 2, 3, 4],
    "seeds_other": [0, 1, 2, 3, 4], "unlabeled_pool": 5000,
    # simulation (synthetic)
    "sim_reps": 8, "sim_budgets": [200, 1000], "sim_sizes": {"source_train": 30000, "source_val": 10000,
                                                             "target_pool": 20000},
    "sim_base_domain": "CA18", "sim_demographic_from": "TX18", "sim_composition_tilt": -0.6,
    "sim_composition_races": [1, 7], "sim_mechanism_low": -1.0, "sim_mechanism_high": 0.3,
    "sim_unselected_tilt": -0.5, "sim_min_mass": 0.003,
    # robustness and trade-off curves
    "robust_budgets": [200, 1000], "robust_seeds": [0, 1, 2, 3, 4], "robust_schemes": ["coarse", "fine"],
    "robust_pooled": [["CA18", "TX18", "NY18"], ["CA18", "NY18", "TX18"], ["TX18", "NY18", "CA18"]],
    "tradeoff_pairs": [["CA18", "TX18"], ["TX18", "CA18"]], "tradeoff_deltas": [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08],
    "tradeoff_seeds": [0, 1, 2, 3, 4],
    # statistics
    "n_boot": 2000, "holm_alpha": 0.05,
}
if RUN_MODE == "quick":
    CFG.update(budgets=[0, 200, 1000], seeds=[0, 1], seeds_other=[0], sim_reps=3, robust_seeds=[0], tradeoff_seeds=[0],
               n_boot=200)
CFG["domains"] = [f"{s}{str(y)[2:]}" for y in CFG["years"] for s in CFG["states"]]
CFG["pairs"] = [tuple(p) for p in CFG["pairs"]]
CFG["sources"] = sorted({s for s, _ in CFG["pairs"]}, key=CFG["domains"].index)
CFG["config_sha256"] = hashlib.sha256(json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:16]

RESULTS = BASE_DIR / ("results" if RUN_MODE == "full" else "results_quick")
PATHS = {"raw": BASE_DIR / "data" / "raw", "processed": BASE_DIR / "data" / "processed", "results": RESULTS,
         "metrics": RESULTS / "metrics", "tables": RESULTS / "tables", "figures": RESULTS / "figures",
         "logs": RESULTS / "logs"}
for _d in PATHS.values():
    _d.mkdir(parents=True, exist_ok=True)
ARTIFACTS, CONTEXT, T_START = [], {"stage": None}, time.time()


def stable_seed(*parts):
    """Deterministic 32-bit seed from strings/ints (independent of Python's hash randomisation)."""
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:8], 16)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _register(path, kind):
    ARTIFACTS.append({"artifact": Path(path).relative_to(BASE_DIR).as_posix(), "kind": kind, "stage": CONTEXT["stage"],
                      "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})


def save_table(df, name, kind="tables", index=False):
    path = PATHS[kind] / f"{name}.csv"
    df.to_csv(path, index=index)
    _register(path, kind)
    return path


def save_json(obj, name, kind="metrics"):
    path = PATHS[kind] / f"{name}.json"
    path.write_text(json.dumps(obj, indent=1, default=str), encoding="utf-8")
    _register(path, kind)
    return path


def save_text(text, name, kind="results"):
    path = PATHS[kind] / name
    path.write_text(text, encoding="utf-8")
    _register(path, kind)
    return path


def save_fig(fig, name):
    for ext in ("png", "pdf"):
        path = PATHS["figures"] / f"{name}.{ext}"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        _register(path, "figures")


def versions():
    out = {"python": sys.version.split()[0], "platform": platform.platform(), "cpu_count": os.cpu_count()}
    for mod in ("numpy", "pandas", "scipy", "sklearn", "matplotlib", "networkx", "pgmpy", "folktables", "fairlearn"):
        try:
            out[mod] = importlib.import_module(mod).__version__
        except Exception:
            out[mod] = "not installed"
    return out


print(f"run mode {RUN_MODE}; data in {PATHS['raw'].parent}; results in {RESULTS}")

### 1.2 Variables, groups and bins
Variables, demographic groups and discretisation schemes (fixed before any model was fitted).

Group index g = 2 * R + X, where R is the ACS race code RAC1P - 1 (9 categories) and X is SEX - 1 (0 = male,
1 = female). Race and sex are self-reported ACS answers; nothing is inferred from other variables.

In [ ]:
RACE_NAMES = ["White", "Black", "AmIndian", "AlaskaNat", "AIAN-oth", "Asian", "NHPI", "OtherRace", "TwoPlus"]
SEX_NAMES = ["M", "F"]
GROUP_NAMES = [f"{r}-{s}" for r in RACE_NAMES for s in SEX_NAMES]           # g = 2*R + X
N_GROUPS = len(GROUP_NAMES)

VARS = ["R", "X", "C", "E", "W", "O", "H", "Y"]                             # BN variables, topological tiers
FEATS = VARS[:-1]                                                          # prediction-time features (incl. R, X)
COVS = ["C", "E", "W", "O", "H"]                                           # non-protected covariates
VAR_LABELS = {"R": "race", "X": "sex", "C": "age", "E": "education", "W": "class of worker", "O": "occupation",
              "H": "hours", "Y": "income>50k"}

OCC_EDGES = [1000, 2000, 3000, 3600, 4700, 6000, 7700]                     # OCCP major-group boundaries
OCC_NAMES = ["Mgmt/Bus/Fin", "Comp/Eng/Sci", "Edu/Legal/Comm/Arts", "Health pract.", "Service", "Sales/Office",
             "Constr/Maint/Farm", "Prod/Transp/Mil"]

# Bin edges (np.digitize, right-open). "primary" is preregistered; "coarse" and "fine" are robustness checks.
SCHEMES = {
    "primary": {"age": [25, 35, 50, 65], "hours": [20, 35, 45], "edu": [16, 18, 21, 22]},
    "coarse": {"age": [30, 45, 60], "hours": [35, 45], "edu": [18, 21]},
    "fine": {"age": [22, 30, 40, 50, 60, 70], "hours": [15, 30, 40, 41, 50], "edu": [12, 16, 18, 20, 21, 22, 23]},
}


def card_of(scheme):
    """Cardinality of every BN variable under a discretisation scheme."""
    s = SCHEMES[scheme]
    return {"R": 9, "X": 2, "C": len(s["age"]) + 1, "E": len(s["edu"]) + 1, "W": 4, "O": len(OCC_EDGES) + 1,
            "H": len(s["hours"]) + 1, "Y": 2}

### 1.3 Data: acquisition, ACSIncome filter and check, encoding, household splits
ACS PUMS acquisition, the Folktables ACSIncome filter and its check, encoding to BN variables, household splits.

Raw zip files go to data/raw and filtered extracts to data/processed (both git-ignored; the Census Bureau terms allow
redistribution, but the repository ships acquisition code and checksums instead of microdata).

In [ ]:
FIPS = {"CA": 6, "TX": 48, "NY": 36}
KEEP_COLS = ["SERIALNO", "SPORDER", "ST", "PWGTP", "AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "WKHP",
             "SEX", "RAC1P", "PINCP"]
REPW_COLS = [f"PWGTP{i}" for i in range(1, 81)]
ACSINCOME_FEATURES = ["AGEP", "COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "WKHP", "SEX", "RAC1P"]
_USE = set(KEEP_COLS + REPW_COLS + ["RELSHIPP"])


def pums_url(state, year):
    return f"https://www2.census.gov/programs-surveys/acs/data/pums/{year}/1-Year/csv_p{state.lower()}.zip"


def download(url, dest, retries=5):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return dest
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ICT5143 capstone notebook)"})
            with urllib.request.urlopen(req, timeout=600) as r, open(dest + ".part", "wb") as f:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
            os.replace(dest + ".part", dest)
            return dest
        except Exception as e:  # network hiccups on the Census server are common; retry with back-off
            print(f"    attempt {attempt} failed: {e}")
            time.sleep(10 * attempt)
    raise RuntimeError(f"Could not download {url}. Re-run this cell; the Census server is sometimes slow.")


def acsincome_mask(df):
    """Folktables `adult_filter` for ACSIncome, written out explicitly."""
    m = (df["AGEP"] > 16) & (df["PINCP"] > 100) & (df["WKHP"] > 0) & (df["PWGTP"] >= 1)
    return m.fillna(False).to_numpy(dtype=bool)


def folktables_check(raw, filt):
    """Compare our filter/label/features with the official folktables ACSIncome problem definition."""
    try:
        from folktables import ACSIncome
    except Exception as e:
        return {"folktables_check": f"skipped ({type(e).__name__}: folktables not installed)"}
    Xf, yf, _ = ACSIncome.df_to_pandas(raw)
    res = {"folktables_rows": int(len(Xf)), "rows_match": bool(len(Xf) == len(filt))}
    if res["rows_match"]:
        ours_y = (filt["PINCP"].to_numpy() > CFG["income_threshold"]).astype(int)
        res["labels_match"] = bool(np.array_equal(yf.iloc[:, 0].to_numpy().astype(int), ours_y))
        feat_ok = True
        for c in ACSINCOME_FEATURES:
            a = Xf[c].to_numpy(float)
            b = filt[c].to_numpy(float)
            ok = ~np.isnan(b)
            feat_ok &= bool(np.allclose(a[ok], b[ok]))
        res["features_match"] = feat_ok
        res["folktables_threshold"] = "PINCP > 50000"
    res["folktables_check"] = "passed" if res.get("rows_match") and res.get("labels_match") and res.get("features_match") else "FAILED"
    return res


def domain_tag(state, year):
    return f"{state}{str(year)[2:]}"


def zip_path(state, year):
    return PATHS["raw"] / f"{year}_csv_p{state.lower()}.zip"


def acquire(state, year):
    """Download one ACS PUMS 1-Year person file (skipped when it is already in data/raw)."""
    path = zip_path(state, year)
    download(pums_url(state, year), str(path))
    return path


def prepare_domain(state, year, force=False):
    """Filter one state-year to the ACSIncome population, check it against Folktables and cache the extract,
    its 80 replicate weights and its provenance record in data/processed."""
    tag = domain_tag(state, year)
    proc = PATHS["processed"]
    cache, cache_w = proc / f"acsincome_{tag}.pkl", proc / f"repweights_{tag}.pkl"
    prov_path = proc / f"provenance_{tag}.json"
    if not force and all(p.exists() for p in (cache, cache_w, prov_path)):
        with open(prov_path) as f:
            return pd.read_pickle(cache), json.load(f)
    url = pums_url(state, year)
    zpath = acquire(state, year)
    t0 = time.time()
    with zipfile.ZipFile(zpath) as z:
        member = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        with z.open(member) as f:
            raw = pd.read_csv(f, usecols=lambda c: c in _USE, dtype={"SERIALNO": str}, low_memory=False)
    relp_note = "RELP"
    if "RELP" not in raw.columns and "RELSHIPP" in raw.columns:
        # From 2019 the Census renamed and recoded RELP as RELSHIPP. Folktables 0.0.12 still expects RELP, so it is
        # aliased only to run the official-definition check. RELP is not used by any model.
        raw["RELP"] = raw["RELSHIPP"]
        relp_note = "RELSHIPP aliased as RELP (2019 rename; recoded; unused by models)"
    mask = acsincome_mask(raw)
    filt = raw.loc[mask].reset_index(drop=True)
    check = folktables_check(raw, filt)
    assert set(filt["ST"].unique()) == {FIPS[state]}, "unexpected state code"
    prov = {"domain": tag, "source_url": url, "zip_member": member, "zip_bytes": zpath.stat().st_size,
            "zip_sha256": sha256_file(zpath), "accessed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "rows_person_file": int(len(raw)), "rows_acsincome": int(len(filt)),
            "households_acsincome": int(filt["SERIALNO"].nunique()),
            "filter": "AGEP>16 & PINCP>100 & WKHP>0 & PWGTP>=1", "target": "PINCP > 50000", "relp": relp_note,
            **check}
    filt[REPW_COLS].astype(np.float32).to_pickle(cache_w)
    filt[KEEP_COLS].to_pickle(cache)
    with open(prov_path, "w") as f:
        json.dump(prov, f, indent=1)
    del raw
    print(f"  {tag}: {len(filt):,} persons ({time.time() - t0:.0f}s), folktables check: {check.get('folktables_check')}")
    return filt[KEEP_COLS], prov


def load_domains():
    """ACSIncome extracts (and provenance records) of every configured state-year, prepared on first use.
    With data_source: synthetic (smoke configuration) the extracts are synthetic stand-ins instead."""
    raw, prov = {}, {}
    for y in CFG["years"]:
        for s in CFG["states"]:
            tag = domain_tag(s, y)
            raw[tag], prov[tag] = prepare_domain(s, y)
    return raw, prov


def load_repweights(tag):
    """The 80 successive-difference replicate weights of a domain's ACSIncome rows (original row order)."""
    return pd.read_pickle(PATHS["processed"] / f"repweights_{tag}.pkl").to_numpy(np.float64)


def encode(df, scheme="primary"):
    """Map raw ACS columns to the discrete BN variables (-1 marks a missing value)."""
    s = SCHEMES[scheme]
    out = {"R": df["RAC1P"].to_numpy(int) - 1, "X": df["SEX"].to_numpy(int) - 1,
           "C": np.digitize(df["AGEP"].to_numpy(float), s["age"]),
           "H": np.digitize(df["WKHP"].to_numpy(float), s["hours"])}
    schl = df["SCHL"].to_numpy(float)
    e = np.digitize(schl, s["edu"])
    e[np.isnan(schl)] = -1
    out["E"] = e
    cow = df["COW"].to_numpy(float)
    w = np.full(len(df), -1)
    w[cow == 1] = 0
    w[cow == 2] = 1
    w[np.isin(cow, [3, 4, 5])] = 2
    w[np.isin(cow, [6, 7, 8])] = 3
    out["W"] = w
    occ = df["OCCP"].to_numpy(float)
    o = np.digitize(occ, OCC_EDGES)
    o[np.isnan(occ) | (occ >= 9920)] = -1
    out["O"] = o
    out["Y"] = (df["PINCP"].to_numpy(float) > CFG["income_threshold"]).astype(int)
    return {k: out[k].astype(np.int8) for k in VARS}


def household_parts(serial, fracs, seed):
    uniq, inv = np.unique(serial, return_inverse=True)
    perm = np.random.default_rng(seed).permutation(len(uniq))
    c1, c2 = int(fracs[0] * len(uniq)), int((fracs[0] + fracs[1]) * len(uniq))
    part_hh = np.empty(len(uniq), np.int8)
    part_hh[perm[:c1]], part_hh[perm[c1:c2]], part_hh[perm[c2:]] = 0, 1, 2
    return part_hh[inv.ravel()], inv.ravel()


def take(data, idx):
    return {k: v[idx] for k, v in data.items()}


def concat(*ds):
    return {k: np.concatenate([d[k] for d in ds]) for k in ds[0]}


def build_domains(raw, scheme="primary"):
    """Encode every domain and split it by household into P1 (train), P2 (validation) and P3 (test).
    The adaptation pool of a target is P1 + P2; its test split P3 never contributes labels or covariates."""
    doms = {}
    for i, tag in enumerate(CFG["domains"]):
        df = raw[tag]
        enc = encode(df, scheme)
        enc["G"] = (2 * enc["R"].astype(int) + enc["X"]).astype(np.int8)
        part, hh = household_parts(df["SERIALNO"].to_numpy().astype(str), CFG["split_fracs"], CFG["split_seed"] + i)
        enc["HH"] = hh.astype(np.int64)                     # household index (for the cluster bootstrap)
        enc["PW"] = df["PWGTP"].to_numpy(float)
        enc["ROW"] = np.arange(len(df))
        d = {"tag": tag, "all": enc, "part": part,
             "idx_train": np.where(part == 0)[0], "idx_val": np.where(part == 1)[0],
             "idx_test": np.where(part == 2)[0], "idx_pool": np.where(part <= 1)[0]}
        d["train"], d["val"], d["test"] = take(enc, d["idx_train"]), take(enc, d["idx_val"]), take(enc, d["idx_test"])
        doms[tag] = d
    return doms


def data_card_tables(doms):
    """Per-domain sizes, missingness and test-split group support; group counts; group positive rates."""
    rows = []
    for tag in CFG["domains"]:
        a, te = doms[tag]["all"], doms[tag]["test"]
        pos_te = np.bincount(te["G"][te["Y"] == 1], minlength=N_GROUPS)
        neg_te = np.bincount(te["G"][te["Y"] == 0], minlength=N_GROUPS)
        rows.append({"domain": tag, "persons": len(a["Y"]), "households": int(a["HH"].max() + 1),
                     "positive rate": a["Y"].mean().round(4),
                     "weighted positive rate": (a["PW"] * a["Y"]).sum() / a["PW"].sum(),
                     "missing E %": 100 * (a["E"] < 0).mean(), "missing W %": 100 * (a["W"] < 0).mean(),
                     "missing O %": 100 * (a["O"] < 0).mean(),
                     "train/val/test": f'{len(doms[tag]["idx_train"])}/{len(doms[tag]["idx_val"])}/{len(te["Y"])}',
                     "groups ≥100 pos & neg (test)": int(((pos_te >= CFG["min_eval"]) & (neg_te >= CFG["min_eval"])).sum()),
                     "groups ≥30 pos & neg (test)": int(((pos_te >= CFG["min_eval_secondary"])
                                                         & (neg_te >= CFG["min_eval_secondary"])).sum())})
    data_card = pd.DataFrame(rows).round(3)
    comp = pd.DataFrame({tag: np.bincount(doms[tag]["all"]["G"], minlength=N_GROUPS) for tag in CFG["domains"]},
                        index=GROUP_NAMES)
    pos_rate = pd.DataFrame({tag: pd.Series(doms[tag]["all"]["Y"]).groupby(doms[tag]["all"]["G"]).mean()
                             .reindex(range(N_GROUPS)).to_numpy() for tag in CFG["domains"]}, index=GROUP_NAMES).round(3)
    return data_card, comp, pos_rate

### 1.4 Bayesian network
Discrete Bayesian network of one domain.

Structure: hill climbing over edges among the covariates C, E, W, O, H with required edges R, X -> every covariate,
tier constraints and an in-degree cap (configs: graph.*), scored by validation log-likelihood. Composition CPTs:
hierarchical Dirichlet smoothing (race row -> race-pooled row -> marginal). Outcome CPD P(Y | R,X,C,E,W,O,H):
L2-regularised log-linear model (main effects + sex interactions), tabulated over all 57,600 parent configurations.

In [ ]:
def edge_allowed(u, v):
    """Tier order (configs: graph.tiers) with the within-tier exceptions in graph.within_tier_allowed."""
    if u == v:
        return False
    tier, within = CFG["tiers"], {tuple(e) for e in CFG["within_tier_allowed"]}
    return tier[u] < tier[v] or (tier[u] == tier[v] and (u, v) in within)


def required_edges():
    """Edges every structure must contain (R -> X and R, X -> each covariate; configs: graph.required_edges)."""
    return {tuple(e) for e in CFG["required_edges"]}


def sort_vars(vs):
    return tuple(sorted(vs, key=VARS.index))


def family_counts(data, child, parents, card, weights=None):
    """(Weighted) counts N[u_1,...,u_k, child] over rows where the whole family is observed."""
    vs = list(parents) + [child]
    arrs = [data[v].astype(np.int64) for v in vs]
    ok = np.ones(len(arrs[0]), bool)
    for a in arrs:
        ok &= a >= 0
    flat = np.zeros(int(ok.sum()), np.int64)
    for v, a in zip(vs, arrs):
        flat = flat * card[v] + a[ok]
    size = int(np.prod([card[v] for v in vs]))
    w = None if weights is None else np.asarray(weights, float)[ok]
    return np.bincount(flat, weights=w, minlength=size).astype(float).reshape([card[v] for v in vs])


def topological_order(parents):
    order, done = [], set()
    while len(order) < len(parents):
        for v in VARS:
            if v not in done and all(p in done for p in parents[v]):
                order.append(v)
                done.add(v)
    return order


def is_ancestor(a, b, parents):
    stack, seen = list(parents[b]), set()
    while stack:
        x = stack.pop()
        if x == a:
            return True
        if x not in seen:
            seen.add(x)
            stack.extend(parents[x])
    return False


def learn_structure(data, card, val=None, score=None, max_indegree=None):
    """Hill climbing (add / delete / reverse) over edges among the covariates, with required R,X edges and tiers.
    score="heldout": log-likelihood of the validation split under the smoothed CPTs fitted on `data` (preregistered);
    score="bic": classical BIC of the maximum-likelihood CPTs (reported for comparison)."""
    score = CFG["structure_score"] if score is None else score
    max_indegree = CFG["max_indegree"] if max_indegree is None else max_indegree
    n = len(data["Y"])
    cache = {}

    def local(v, pa):
        if (v, pa) not in cache:
            if score == "heldout":
                cache[(v, pa)] = float((family_counts(val, v, pa, card) * np.log(fit_cpt(data, v, pa, card))).sum())
            else:
                cnt = family_counts(data, v, pa, card).reshape(-1, card[v])
                n_u = cnt.sum(1, keepdims=True)
                with np.errstate(divide="ignore", invalid="ignore"):
                    ll = np.where(cnt > 0, cnt * np.log(cnt / n_u), 0.0).sum()
                cache[(v, pa)] = ll - 0.5 * np.log(n) * (card[v] - 1) * cnt.shape[0]
        return cache[(v, pa)]

    parents = {v: () for v in VARS}
    for u, v in required_edges():
        parents[v] = sort_vars(set(parents[v]) | {u})
    history = []
    while True:
        best_gain, best_op = 1e-9, None
        for v in COVS:
            for u in COVS:
                if u == v:
                    continue
                if u in parents[v]:
                    pa_del = sort_vars(set(parents[v]) - {u})
                    gain = local(v, pa_del) - local(v, parents[v])
                    if gain > best_gain:
                        best_gain, best_op = gain, ("delete", u, v)
                    if edge_allowed(v, u) and len(parents[u]) < max_indegree:
                        tmp = dict(parents)
                        tmp[v] = pa_del
                        if not is_ancestor(u, v, tmp):
                            g2 = gain + local(u, sort_vars(set(parents[u]) | {v})) - local(u, parents[u])
                            if g2 > best_gain:
                                best_gain, best_op = g2, ("reverse", u, v)
                elif edge_allowed(u, v) and len(parents[v]) < max_indegree and not is_ancestor(v, u, parents):
                    gain = local(v, sort_vars(set(parents[v]) | {u})) - local(v, parents[v])
                    if gain > best_gain:
                        best_gain, best_op = gain, ("add", u, v)
        if best_op is None:
            break
        op, u, v = best_op
        if op in ("delete", "reverse"):
            parents[v] = sort_vars(set(parents[v]) - {u})
        if op == "add":
            parents[v] = sort_vars(set(parents[v]) | {u})
        if op == "reverse":
            parents[u] = sort_vars(set(parents[u]) | {v})
        history.append((op, u, v, round(best_gain, 1)))
    parents["Y"] = tuple(FEATS)                       # log-linear outcome CPD over all features
    return parents, history, sum(local(v, parents[v]) for v in VARS if v != "Y")


def fit_cpt(data, child, parents, card, alpha0=None, alpha_race=None, weights=None):
    """Hierarchical Dirichlet (m-estimate) smoothing: race rows -> race-pooled row -> marginal."""
    alpha0 = CFG["alpha0"] if alpha0 is None else alpha0
    alpha_race = CFG["alpha_race"] if alpha_race is None else alpha_race
    cnt = family_counts(data, child, parents, card, weights)
    if "R" in parents and len(parents) > 1:
        pooled = fit_cpt(data, child, tuple(p for p in parents if p != "R"), card, alpha0, alpha_race, weights)
        base, a = np.expand_dims(pooled, parents.index("R")), alpha_race
    else:
        m = family_counts(data, child, (), card, weights) + 0.5
        base, a = m / m.sum(), alpha0
    return (cnt + a * base) / (cnt.sum(-1, keepdims=True) + a)


_CONFIG_CACHE = {}


def all_configs(card):
    """All feature configurations in C order over FEATS (the parent configurations of the outcome CPD)."""
    key = tuple(card[v] for v in FEATS)
    if key not in _CONFIG_CACHE:
        _CONFIG_CACHE[key] = np.indices(key).reshape(len(FEATS), -1).T.astype(np.int16)
    return _CONFIG_CACHE[key]


def main_effects(X, card):
    """Dense [intercept, one-hot main effects] design used by the outcome-factor update (37 + 1 columns)."""
    cols = [np.ones((len(X), 1))]
    for j, v in enumerate(FEATS):
        oh = np.zeros((len(X), card[v]))
        ok = X[:, j] >= 0
        oh[np.where(ok)[0], X[ok, j]] = 1.0
        cols.append(oh)
    return np.hstack(cols)


class OutcomeModel:
    """Log-linear CPD P(Y=1|x) = sigmoid(w·phi(x)); phi = one-hot main effects + sex × one-hot interactions."""

    def __init__(self, card, C=None):
        self.card, self.C = card, (CFG["outcome_l2_C"] if C is None else C)
        self.enc = OneHotEncoder(categories=[np.arange(card[v]) for v in FEATS], handle_unknown="ignore")
        self.enc.fit(np.zeros((1, len(FEATS)), np.int16))

    def design(self, X):
        from scipy import sparse
        base = self.enc.transform(X)
        female = sparse.diags((X[:, FEATS.index("X")] == 1).astype(float))
        return sparse.hstack([base, female @ base]).tocsr()

    def fit(self, X, y, w=None, init=None):
        self.m = LogisticRegression(C=self.C, max_iter=5000, warm_start=init is not None)
        if init is not None:
            self.m.coef_, self.m.intercept_ = init.m.coef_.copy(), init.m.intercept_.copy()
            self.m.classes_ = init.m.classes_.copy()
        self.m.fit(self.design(X), y, sample_weight=w)
        return self

    def proba(self, X, D=None):
        return self.m.predict_proba(self.design(X) if D is None else D)[:, 1]

    def table(self):
        X = all_configs(self.card)
        p = self.proba(X)
        return np.stack([1 - p, p], -1).reshape([self.card[v] for v in FEATS] + [2])


def feats_int(data):
    return np.stack([data[v] for v in FEATS], 1).astype(np.int16)


class BayesNet:
    """Discrete BN over VARS; cpts[v] has shape (card[p1], ..., card[pk], card[v]) with parents sorted as in VARS."""

    def __init__(self, parents, card, cpts):
        self.parents = {v: tuple(parents[v]) for v in VARS}
        self.card = dict(card)
        self.cpts = {v: np.asarray(cpts[v], float) for v in VARS}
        self.order = topological_order(self.parents)

    def copy(self, **new_cpts):
        c = {v: self.cpts[v].copy() for v in VARS}
        c.update(new_cpts)
        return BayesNet(self.parents, self.card, c)

    def scope(self, v):
        return self.parents[v] + (v,)

    def n_params(self):
        return {v: int(self.cpts[v].size // self.card[v] * (self.card[v] - 1)) for v in VARS}

    def check(self, tol=1e-9):
        for v in VARS:
            assert self.cpts[v].shape == tuple(self.card[u] for u in self.scope(v)), f"bad CPT shape for {v}"
            assert np.all(self.cpts[v] > 0), f"zero probability in CPT of {v}"
            err = np.abs(self.cpts[v].sum(-1) - 1).max()
            assert err < tol, f"CPT of {v} not normalised ({err})"
        return True

    def joint(self):
        """Exact joint P(R,X,C,E,W,O,H,Y) by enumeration (one axis per variable, VARS order)."""
        shape = [self.card[v] for v in VARS]
        J = np.ones(shape)
        for v in VARS:
            axes = [VARS.index(u) for u in self.scope(v)]
            arr = np.transpose(self.cpts[v], np.argsort(axes))
            bshape = [1] * len(VARS)
            for ax in sorted(axes):
                bshape[ax] = shape[ax]
            J = J * arr.reshape(bshape)
        return J

    def factor_loglik(self, data, v):
        arrs = [data[u].astype(np.int64) for u in self.scope(v)]
        ok = np.all([a >= 0 for a in arrs], axis=0)
        out = np.full(len(arrs[0]), np.nan)
        out[ok] = np.log(self.cpts[v][tuple(a[ok] for a in arrs)])
        return out

    def predict_proba(self, data, J=None):
        """P(Y=1 | observed features): table lookup when every parent of Y is observed, exact marginalisation otherwise."""
        pa = self.parents["Y"]
        th = self.cpts["Y"][..., 1]
        arrs = [data[u].astype(np.int64) for u in pa]
        ok = np.all([a >= 0 for a in arrs], axis=0)
        p = np.full(len(arrs[0]), np.nan)
        p[ok] = th[tuple(a[ok] for a in arrs)]
        miss = np.where(~ok)[0]
        if len(miss):
            J = self.joint() if J is None else J
            keys = np.stack([data[v][miss].astype(int) for v in FEATS], 1)
            uniq, inv = np.unique(keys, axis=0, return_inverse=True)
            vals = np.empty(len(uniq))
            for i, row in enumerate(uniq):
                sub = J[tuple(int(x) if x >= 0 else slice(None) for x in row) + (slice(None),)].reshape(-1, 2).sum(0)
                vals[i] = sub[1] / sub.sum()
            p[miss] = vals[inv.ravel()]
        return p

    def sample(self, n, rng):
        out = {}
        for v in self.order:
            pa, K = self.parents[v], self.card[v]
            th = self.cpts[v].reshape(-1, K)
            idx = np.ravel_multi_index([out[u] for u in pa], [self.card[u] for u in pa]) if pa else np.zeros(n, np.int64)
            cdf = np.cumsum(th, axis=1)[idx]
            out[v] = np.minimum((rng.random(n)[:, None] > cdf).sum(1), K - 1).astype(np.int8)
        data = {v: out[v] for v in VARS}
        data["G"] = (2 * data["R"].astype(int) + data["X"]).astype(np.int8)
        return data


def fit_bn(train, parents, card, outcome=None, weights=None):
    """Composition CPTs by hierarchical smoothing; outcome CPD = (given or newly fitted) log-linear model, tabulated."""
    if outcome is None:
        outcome = OutcomeModel(card).fit(feats_int(train), train["Y"], w=weights)
    cpts = {v: fit_cpt(train, v, parents[v], card, weights=weights) for v in VARS if v != "Y"}
    cpts["Y"] = outcome.table()
    bn = BayesNet(parents, card, cpts)
    bn.check()
    return bn, outcome


def describe_structure(parents):
    return "  ".join(f"P({v}|{','.join(parents[v])})" if parents[v] else f"P({v})" for v in VARS)

### 1.5 Exact inference
Exact inference on the discrete BN.

Factor algebra; variable elimination with min-fill, min-degree, min-weight and random orders; a clique tree built from
an elimination order with two-pass sum-product calibration; brute-force enumeration of the joint; a pgmpy cross-check;
d-separation by the moralised ancestral graph; and G-tests (with bias-corrected conditional mutual information) of the
local Markov statements implied by a learned graph.

In [ ]:
class Factor:
    __slots__ = ("vars", "card", "values")

    def __init__(self, vars_, card, values):
        self.vars, self.card = tuple(vars_), tuple(int(c) for c in card)
        self.values = np.asarray(values, float).reshape(self.card)

    def _expand(self, order, cmap):
        perm = [self.vars.index(v) for v in order if v in self.vars]
        return np.transpose(self.values, perm).reshape([cmap[v] if v in self.vars else 1 for v in order])

    def product(self, other):
        order = list(self.vars) + [v for v in other.vars if v not in self.vars]
        cmap = dict(zip(self.vars, self.card))
        cmap.update(zip(other.vars, other.card))
        return Factor(order, [cmap[v] for v in order], self._expand(order, cmap) * other._expand(order, cmap))

    def marginalize(self, var):
        ax = self.vars.index(var)
        return Factor(self.vars[:ax] + self.vars[ax + 1:], self.card[:ax] + self.card[ax + 1:], self.values.sum(axis=ax))

    def reduce(self, evidence):
        if not any(v in evidence for v in self.vars):
            return self
        idx = tuple(evidence[v] if v in evidence else slice(None) for v in self.vars)
        keep = [i for i, v in enumerate(self.vars) if v not in evidence]
        return Factor([self.vars[i] for i in keep], [self.card[i] for i in keep], self.values[idx])

    def transpose(self, order):
        perm = [self.vars.index(v) for v in order]
        return Factor(order, [self.card[i] for i in perm], np.transpose(self.values, perm))


def bn_factors(bn, include_y=True):
    return [Factor(bn.scope(v), [bn.card[u] for u in bn.scope(v)], bn.cpts[v]) for v in VARS if include_y or v != "Y"]


def interaction_graph(factors):
    G = nx.Graph()
    for f in factors:
        G.add_nodes_from(f.vars)
        G.add_edges_from(itertools.combinations(f.vars, 2))
    return G


def elimination_order(factors, elim_vars, heuristic="min_fill", rng=None):
    G = interaction_graph(factors)
    cmap = {}
    for f in factors:
        cmap.update(zip(f.vars, f.card))
    remaining, order = set(elim_vars), []
    while remaining:
        def cost(v):
            nb = list(G.neighbors(v))
            if heuristic == "min_degree":
                return len(nb)
            if heuristic == "min_weight":
                return float(np.prod([cmap[u] for u in nb + [v]]))
            return sum(1 for a, b in itertools.combinations(nb, 2) if not G.has_edge(a, b))   # min_fill
        cand = sorted(remaining, key=VARS.index)
        v = cand[rng.integers(len(cand))] if heuristic == "random" else min(cand, key=cost)
        nb = list(G.neighbors(v))
        G.add_edges_from(itertools.combinations(nb, 2))
        G.remove_node(v)
        order.append(v)
        remaining.remove(v)
    return order


def variable_elimination(factors, query, evidence=None, heuristic="min_fill", order=None, rng=None, stats=None):
    evidence = evidence or {}
    fs = [f.reduce(evidence) for f in factors]
    present = sort_vars(set(itertools.chain.from_iterable(f.vars for f in fs)))
    order = order or elimination_order(fs, [v for v in present if v not in query], heuristic, rng)
    peak, ops = 0, 0
    for v in order:
        rel = [f for f in fs if v in f.vars]
        if not rel:
            continue
        fs = [f for f in fs if v not in f.vars]
        prod = rel[0]
        for f in rel[1:]:
            prod = prod.product(f)
            ops += prod.values.size
        peak = max(peak, prod.values.size)
        ops += prod.values.size
        fs.append(prod.marginalize(v))
    res = fs[0]
    for f in fs[1:]:
        res = res.product(f)
    res = res.transpose(list(query))
    if stats is not None:
        stats.update(order=order, peak_factor=peak, ops=ops)
    return res.values / res.values.sum()


class CliqueTree:
    """Clique tree built from an elimination order; two-pass sum-product calibration."""

    def __init__(self, factors, order):
        G = interaction_graph(factors)
        self.cmap = {}
        for f in factors:
            self.cmap.update(zip(f.vars, f.card))
        cliques, H = [], G.copy()
        for v in order:
            nb = set(H.neighbors(v))
            cliques.append(frozenset(nb | {v}))
            H.add_edges_from(itertools.combinations(nb, 2))
            H.remove_node(v)
        self.cliques = list(dict.fromkeys(c for c in cliques if not any(c < d for d in cliques)))
        CG = nx.Graph()
        CG.add_nodes_from(range(len(self.cliques)))
        for i, j in itertools.combinations(range(len(self.cliques)), 2):
            w = len(self.cliques[i] & self.cliques[j])
            if w:
                CG.add_edge(i, j, weight=w)
        self.tree = nx.maximum_spanning_tree(CG)
        self.assign = {i: [] for i in range(len(self.cliques))}
        for f in factors:
            self.assign[next(i for i, c in enumerate(self.cliques) if set(f.vars) <= c)].append(f)

    def calibrate(self, evidence=None):
        evidence = evidence or {}
        pot, placed = {}, set()
        for i, c in enumerate(self.cliques):
            vs = sort_vars(c)
            phi = Factor(vs, [self.cmap[v] for v in vs], np.ones([self.cmap[v] for v in vs]))
            for f in self.assign[i]:
                phi = phi.product(f)
            for e, val in evidence.items():
                if e in c and e not in placed:
                    ind = np.zeros(self.cmap[e])
                    ind[val] = 1.0
                    phi = phi.product(Factor([e], [self.cmap[e]], ind))
                    placed.add(e)
            pot[i] = phi.transpose(vs)
        msgs = {}
        for comp in nx.connected_components(self.tree):
            root = min(comp)
            parent = {root: None}
            for u, v in nx.dfs_edges(self.tree, root):
                parent[v] = u
            for i in nx.dfs_postorder_nodes(self.tree, root):
                if parent[i] is not None:
                    msgs[(i, parent[i])] = self._message(i, parent[i], pot, msgs)
            for u, v in nx.dfs_edges(self.tree, root):
                msgs[(u, v)] = self._message(u, v, pot, msgs)
        self.beliefs = {}
        for i in range(len(self.cliques)):
            b = pot[i]
            for k in self.tree.neighbors(i):
                b = b.product(msgs[(k, i)])
            self.beliefs[i] = b
        return self

    def _message(self, i, j, pot, msgs):
        b = pot[i]
        for k in self.tree.neighbors(i):
            if k != j:
                b = b.product(msgs[(k, i)])
        for v in list(b.vars):
            if v not in self.cliques[i] & self.cliques[j]:
                b = b.marginalize(v)
        return b

    def marginal(self, var):
        b = self.beliefs[next(i for i, c in enumerate(self.cliques) if var in c)]
        for v in list(b.vars):
            if v != var:
                b = b.marginalize(v)
        return b.values / b.values.sum()

    def max_calibration_error(self):
        err = 0.0
        for i, j in self.tree.edges():
            sep = sort_vars(self.cliques[i] & self.cliques[j])
            m = []
            for k in (i, j):
                b = self.beliefs[k]
                for v in list(b.vars):
                    if v not in sep:
                        b = b.marginalize(v)
                b = b.transpose(sep).values
                m.append(b / b.sum())
            err = max(err, float(np.abs(m[0] - m[1]).max()))
        return err


def enumeration_query(J, var, evidence, names=VARS):
    sub = J[tuple(evidence[v] if v in evidence else slice(None) for v in names)]
    kept = [v for v in names if v not in evidence]
    p = sub.sum(axis=tuple(i for i, v in enumerate(kept) if v != var))
    return p / p.sum()


def to_pgmpy(bn, include_y=True):
    try:
        from pgmpy.models import DiscreteBayesianNetwork as PGBN
    except ImportError:
        from pgmpy.models import BayesianNetwork as PGBN
    from pgmpy.factors.discrete import TabularCPD
    nodes = [v for v in VARS if include_y or v != "Y"]
    m = PGBN([(p, v) for v in nodes for p in bn.parents[v]])
    m.add_nodes_from(nodes)
    for v in nodes:
        pa = list(bn.parents[v])
        m.add_cpds(TabularCPD(v, bn.card[v], bn.cpts[v].reshape(-1, bn.card[v]).T, evidence=pa or None,
                              evidence_card=[bn.card[p] for p in pa] or None))
    m.check_model()
    return m


def run_inference_checks(bn, include_y, n_queries=20, seed=0, test=None):
    rng = np.random.default_rng(seed)
    names = VARS if include_y else FEATS
    fs = bn_factors(bn, include_y)
    J = bn.joint() if include_y else bn.joint().sum(-1)
    out = {"max |CPT row sum - 1|": max(float(np.abs(bn.cpts[v].sum(-1) - 1).max()) for v in names),
           "|sum of joint - 1|": float(abs(J.sum() - 1))}
    try:
        from pgmpy.inference import VariableElimination as PGVE, BeliefPropagation as PGBP
        pgm = to_pgmpy(bn, include_y)
        pg_ve, pg_bp = PGVE(pgm), PGBP(pgm)
        pg_bp.calibrate()
    except Exception as e:
        pg_ve = pg_bp = None
        out["pgmpy"] = f"skipped ({type(e).__name__}: {e})"
    diffs = {k: 0.0 for k in ["VE min_degree", "VE min_weight", "VE random order", "clique tree",
                              "clique-tree calibration", "enumeration", "pgmpy VE", "pgmpy BP"]}
    n_cliques = []
    for q in range(n_queries):
        k = int(rng.integers(0, 4))
        cand = [v for v in names if v != "Y"]
        ev = {v: int(rng.integers(bn.card[v])) for v in (rng.choice(cand, size=k, replace=False) if k else [])}
        target = "Y" if (include_y and q % 2 == 0) else str(rng.choice([v for v in names if v not in ev]))
        ref = variable_elimination(fs, [target], ev, "min_fill")
        for h, key in [("min_degree", "VE min_degree"), ("min_weight", "VE min_weight"), ("random", "VE random order")]:
            diffs[key] = max(diffs[key], np.abs(variable_elimination(fs, [target], ev, h, rng=rng) - ref).max())
        ct = CliqueTree(fs, elimination_order(fs, list(names), "min_fill")).calibrate(ev)
        n_cliques.append(len(ct.cliques))
        diffs["clique tree"] = max(diffs["clique tree"], np.abs(ct.marginal(target) - ref).max())
        diffs["clique-tree calibration"] = max(diffs["clique-tree calibration"], ct.max_calibration_error())
        diffs["enumeration"] = max(diffs["enumeration"], np.abs(enumeration_query(J, target, ev, names) - ref).max())
        if pg_ve is not None:
            r1 = pg_ve.query([target], evidence=ev or None, show_progress=False).values
            r2 = pg_bp.query([target], evidence=ev or None, show_progress=False).values
            diffs["pgmpy VE"] = max(diffs["pgmpy VE"], np.abs(r1 - ref).max())
            diffs["pgmpy BP"] = max(diffs["pgmpy BP"], np.abs(r2 - ref).max())
    if include_y and test is not None:
        p_fast = bn.predict_proba(test)
        rows_ = rng.choice(len(test["Y"]), 30, replace=False)
        diffs["fast predict_proba"] = max(abs(variable_elimination(fs, ["Y"], {v: int(test[v][i]) for v in FEATS})[1]
                                              - p_fast[i]) for i in rows_)
        miss = take(test, rows_)
        for v in ("O", "E"):
            miss[v] = miss[v].copy()
            miss[v][::2] = -1                                 # delete values to exercise exact marginalisation
        p_miss = bn.predict_proba(miss, J=J)
        diffs["missing-value marginalisation"] = max(
            abs(variable_elimination(fs, ["Y"], {v: int(miss[v][k]) for v in FEATS if miss[v][k] >= 0})[1] - p_miss[k])
            for k in range(len(rows_)))
    if pg_ve is None:
        diffs.pop("pgmpy VE"), diffs.pop("pgmpy BP")
    out.update({f"max abs diff: {k}": float(v) for k, v in diffs.items()})
    out["cliques in clique tree"] = float(np.mean(n_cliques))
    return out


def d_separated(parents, xs, ys, zs):
    xs, ys, zs = set(xs), set(ys), set(zs)
    anc, stack = set(), list(xs | ys | zs)
    while stack:
        v = stack.pop()
        if v not in anc:
            anc.add(v)
            stack.extend(parents[v])
    M = nx.Graph()
    M.add_nodes_from(anc)
    for v in anc:
        pa = [p for p in parents[v] if p in anc]
        M.add_edges_from((p, v) for p in pa)
        M.add_edges_from(itertools.combinations(pa, 2))
    M.remove_nodes_from(zs)
    return not any(nx.has_path(M, x, y) for x in xs for y in ys if x in M and y in M)


def ci_test(data, a, b, cond, card):
    cnt = family_counts(data, b, [a] + list(cond), card)          # axes: a, cond..., b
    cnt = np.moveaxis(cnt, -1, 1).reshape(card[a], card[b], -1)
    n_z, n_az, n_bz = cnt.sum((0, 1)), cnt.sum(1), cnt.sum(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        expct = n_az[:, None, :] * n_bz[None, :, :] / n_z[None, None, :]
        g2 = 2 * np.nansum(np.where(cnt > 0, cnt * np.log(cnt / expct), 0.0))
    df = int(sum(((n_az[:, k] > 0).sum() - 1) * ((n_bz[:, k] > 0).sum() - 1) for k in np.where(n_z > 0)[0]))
    n_tot = cnt.sum()
    # plug-in CMI is biased upward by about df/(2N) nats under independence; report the excess over that expectation
    return (g2, max(df, 1), float(stats.chi2.sf(g2, max(df, 1))), g2 / (2 * n_tot) / np.log(2),
            max(g2 - df, 0.0) / (2 * n_tot) / np.log(2))

### 1.6 Fairness metrics and equalized-odds post-processing
Group fairness metrics, calibration, and equalized-odds threshold post-processing. Definitions over the race × sex
groups g with enough support in the evaluation split:

| Metric | Definition | Meaning |
|---|---|---|
| `d_dp` | max_g SR_g − min_g SR_g | demographic parity gap (SR = selection rate P(D=1 \| g)) |
| `d_eopp` | max_g TPR_g − min_g TPR_g | equal opportunity gap |
| `d_fpr` | max_g FPR_g − min_g FPR_g | false-positive-rate gap |
| `d_eodds` | max(d_eopp, d_fpr) | equalized odds gap (**primary** fairness metric) |
| `r_sel` | min_g SR_g / max_g SR_g | selection-rate ratio (NaN = undefined when every SR_g is 0) |

* Directions: gaps lower is fairer; `ba`, `auroc`, `ap` higher is better; `logloss`, `brier`, `ece` lower is better.
* D = 1 means "predicted income above \$50,000" (a favourable classification, e.g. for credit or service eligibility).
* Primary support: ≥ 100 positives **and** negatives in the evaluated split; secondary: ≥ 30; smaller groups are
  reported, but only as exploratory.


In [ ]:
THRESHOLD_EVENTS = Counter()        # counts of threshold-selection fallbacks (written to the run log)


def support_groups(y, g, min_n):
    pos = np.bincount(g[y == 1].astype(np.int64), minlength=N_GROUPS)
    neg = np.bincount(g[y == 0].astype(np.int64), minlength=N_GROUPS)
    return np.where((pos >= min_n) & (neg >= min_n))[0]


def _gaps(tpr, fpr, sel, support):
    s = np.asarray(support, bool)
    if s.sum() < 2:
        return {k: np.nan for k in ["d_eodds", "d_eopp", "d_fpr", "d_dp", "r_sel", "worst_tpr"]}
    d_tpr, d_fpr = tpr[s].max() - tpr[s].min(), fpr[s].max() - fpr[s].min()
    return {"d_eodds": max(d_tpr, d_fpr), "d_eopp": d_tpr, "d_fpr": d_fpr, "d_dp": sel[s].max() - sel[s].min(),
            "r_sel": (sel[s].min() / sel[s].max()) if sel[s].max() > 0 else np.nan, "worst_tpr": tpr[s].min()}


def group_counts(y, d, g, w=None):
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    g = g.astype(np.int64)
    yp, dp = (y == 1), (d == 1)
    return (np.bincount(g, w * yp, N_GROUPS), np.bincount(g, w * ~yp, N_GROUPS),
            np.bincount(g, w * (yp & dp), N_GROUPS), np.bincount(g, w * (~yp & dp), N_GROUPS))


def ece_by_group(y, s, g, support, bins=10):
    b = np.minimum((s * bins).astype(int), bins - 1)
    key = g.astype(np.int64) * bins + b
    n = np.bincount(key, minlength=N_GROUPS * bins).reshape(N_GROUPS, bins)
    sy = np.bincount(key, y.astype(float), N_GROUPS * bins).reshape(N_GROUPS, bins)
    ss = np.bincount(key, s, N_GROUPS * bins).reshape(N_GROUPS, bins)
    ece = (np.abs(sy - ss).sum(1)) / np.maximum(n.sum(1), 1)
    return ece[np.asarray(support, bool)]


def _evaluate_plain(y, s, d, g, w=None, min_eval=None, scores_metrics=True):
    min_eval = CFG["min_eval"] if min_eval is None else min_eval
    pos, neg, tp, fp = group_counts(y, d, g, w)
    upos = np.bincount(g[y == 1].astype(np.int64), minlength=N_GROUPS)
    uneg = np.bincount(g[y == 0].astype(np.int64), minlength=N_GROUPS)
    support = (upos >= min_eval) & (uneg >= min_eval)
    with np.errstate(divide="ignore", invalid="ignore"):
        tpr, fpr, sel = tp / pos, fp / neg, (tp + fp) / (pos + neg)
    m = _gaps(tpr, fpr, sel, support)
    sec = (upos >= CFG["min_eval_secondary"]) & (uneg >= CFG["min_eval_secondary"])
    m2 = _gaps(tpr, fpr, sel, sec)
    m.update({"d_eodds_30": m2["d_eodds"], "d_dp_30": m2["d_dp"], "n_eval_groups_30": int(sec.sum())})
    P, N, TP, FP = pos.sum(), neg.sum(), tp.sum(), fp.sum()
    m.update({"ba": 0.5 * (TP / P + 1 - FP / N), "acc": (TP + N - FP) / (P + N), "sel_rate": (TP + FP) / (P + N),
              "precision": TP / max(TP + FP, 1e-12), "n_eval_groups": int(support.sum())})
    for name, key in (("sex", np.arange(N_GROUPS) % 2), ("race", np.arange(N_GROUPS) // 2)):
        k = key.max() + 1
        agg = [np.bincount(key, a, k) for a in (pos, neg, tp, fp, upos, uneg)]
        with np.errstate(divide="ignore", invalid="ignore"):
            mm = _gaps(agg[2] / agg[0], agg[3] / agg[1], (agg[2] + agg[3]) / (agg[0] + agg[1]),
                       (agg[4] >= min_eval) & (agg[5] >= min_eval))
        m[f"d_eodds_{name}"] = mm["d_eodds"]
    if scores_metrics:
        sc = np.clip(s, 1e-6, 1 - 1e-6)
        m["auroc"] = roc_auc_score(y, s, sample_weight=w)
        m["ap"] = average_precision_score(y, s, sample_weight=w)
        ww = np.ones(len(y)) if w is None else w
        m["logloss"] = float(-(ww * (y * np.log(sc) + (1 - y) * np.log(1 - sc))).sum() / ww.sum())
        m["brier"] = float((ww * (sc - y) ** 2).sum() / ww.sum())
        e = ece_by_group(y, sc, g, support)
        m["max_group_ece"] = float(e.max()) if len(e) else np.nan
    return m


def evaluate(y, s, d, g, w=None, min_eval=None, scores_metrics=True, exact=None):
    """Utility, fairness and probabilistic metrics of scores `s` and decisions `d` on an evaluation split.
    exact=(weights, support): exact evaluation on an enumerated test set whose rows are every configuration and whose
    weights are the true probabilities (simulation); `support` then selects the groups with enough probability mass."""
    if exact is not None:
        w_exact, support = exact
        m = _evaluate_plain(y, s, d, g, w=w_exact, min_eval=0, scores_metrics=scores_metrics)
        pos, neg, tp, fp = group_counts(y, d, g, w_exact)
        with np.errstate(divide="ignore", invalid="ignore"):
            m.update(_gaps(tp / pos, fp / neg, (tp + fp) / (pos + neg), support))
        m.update({"n_eval_groups": int(support.sum()), "d_eodds_30": m["d_eodds"]})
        return m
    return _evaluate_plain(y, s, d, g, w=w, min_eval=min_eval, scores_metrics=scores_metrics)


def make_grid(scores, masses, max_c=400):
    s = np.concatenate(scores)
    u = np.unique(s)
    if len(u) > max_c:
        m = np.concatenate(masses)
        o = np.argsort(s, kind="stable")
        cw = np.cumsum(m[o])
        cw = cw / cw[-1] if cw[-1] > 0 else np.linspace(0, 1, len(cw))
        u = np.unique(s[o][np.clip(np.searchsorted(cw, np.linspace(0, 1, max_c)), 0, len(s) - 1)])
    return np.append(u, np.inf)


def roc_counts(s, m1, m0, grid):
    o = np.argsort(s, kind="stable")
    c1 = np.concatenate([[0.0], np.cumsum(m1[o])])
    c0 = np.concatenate([[0.0], np.cumsum(m0[o])])
    k = np.searchsorted(s[o], grid, side="left")          # scores strictly below the threshold
    return c1[-1] - c1[k], c0[-1] - c0[k], c1[-1], c0[-1]


def select_thresholds(tables, fit_groups, delta=None, mode="eo", fallback=None):
    """Group thresholds from per-group (scores, positive mass, negative mass) tables. Returns (thresholds[18], info).
    Groups outside `fit_groups` keep the pooled threshold, or `fallback[g]` when a fallback rule is given."""
    delta = CFG["delta_ba"] if delta is None else delta
    grid = make_grid([t[0] for t in tables], [t[1] + t[2] for t in tables])
    C = len(grid)
    TP, FP, P, Nn = np.zeros((N_GROUPS, C)), np.zeros((N_GROUPS, C)), np.zeros(N_GROUPS), np.zeros(N_GROUPS)
    for g, (s, m1, m0) in enumerate(tables):
        if len(s):
            TP[g], FP[g], P[g], Nn[g] = roc_counts(s, m1, m0, grid)
    P0, N0 = P.sum(), Nn.sum()
    tpr0, fpr0 = TP.sum(0) / P0, FP.sum(0) / N0
    ba0 = 0.5 * (tpr0 + 1 - fpr0)
    j0 = int(np.argmax(ba0))
    fit = np.asarray(fit_groups, int)
    if (mode == "single" or len(fit) < 2) and fallback is None:
        if mode != "single":
            THRESHOLD_EVENTS["fewer than 2 supported groups: pooled threshold"] += 1
        return np.full(N_GROUPS, grid[j0]), {"est_ba": ba0[j0], "est_deo": np.nan, "tau0": grid[j0],
                                             "n_fit": len(fit), "mode": "single"}
    if len(fit) < 2:                                       # nothing to equalise: keep the fallback rule
        THRESHOLD_EVENTS["fewer than 2 supported groups: fallback thresholds kept"] += 1
        return np.asarray(fallback, float).copy(), {"est_ba": np.nan, "est_deo": np.nan, "tau0": np.nan,
                                                    "n_fit": len(fit), "mode": "fallback"}
    tg, fg = TP[fit] / P[fit, None], FP[fit] / Nn[fit, None]
    D = np.maximum(np.abs(tg[:, None, :] - tpr0[None, :, None]), np.abs(fg[:, None, :] - fpr0[None, :, None]))
    kb = D.argmin(2)
    ts, fs_ = np.take_along_axis(tg, kb, 1), np.take_along_axis(fg, kb, 1)
    deo = np.maximum(ts.max(0) - ts.min(0), fs_.max(0) - fs_.min(0))
    other = np.setdiff1d(np.arange(N_GROUPS), fit)
    if fallback is None:
        tp_oth, fp_oth = TP[other].sum(0), FP[other].sum(0)
    else:                                                  # unsupported groups keep their fallback thresholds
        tp_oth = fp_oth = 0.0
        for g in other:
            s, m1, m0 = tables[g]
            if len(s):
                a, b, _, _ = roc_counts(s, m1, m0, np.array([fallback[g]]))
                tp_oth, fp_oth = tp_oth + a[0], fp_oth + b[0]
    ba = 0.5 * (((ts * P[fit, None]).sum(0) + tp_oth) / P0 + 1 - ((fs_ * Nn[fit, None]).sum(0) + fp_oth) / N0)
    feas = np.where(ba >= ba0[j0] - delta - 1e-12)[0]
    if len(feas) == 0:                      # no equalised point within the utility tolerance: keep the best-utility one
        feas = np.array([int(np.argmax(ba))])
        THRESHOLD_EVENTS["no equalised point within delta: best-BA point used"] += 1
    j = feas[np.lexsort((-ba[feas], deo[feas]))[0]]
    thr = np.full(N_GROUPS, grid[j]) if fallback is None else np.asarray(fallback, float).copy()
    thr[fit] = grid[kb[:, j]]
    return thr, {"est_ba": ba[j], "est_deo": deo[j], "est_ba_max": ba0[j0], "tau0": grid[j], "n_fit": len(fit),
                 "mode": "eo"}


def data_tables(s, y, g, w=None):
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    out = []
    for k in range(N_GROUPS):
        m = g == k
        out.append((s[m], w[m] * (y[m] == 1), w[m] * (y[m] == 0)))
    return out


def model_tables(bn_mass, bn_score=None, J=None):
    """Exact per-group tables (p(u), P(G=g, pa(Y)=u, Y=1), P(G=g, pa(Y)=u, Y=0)) from the BN joint.
    Masses come from `bn_mass` (e.g. an adapted model of the target); scores from `bn_score` (the rule being audited)."""
    bn_score = bn_mass if bn_score is None else bn_score
    pa = bn_mass.parents["Y"]
    assert bn_score.parents["Y"] == pa
    J = bn_mass.joint() if J is None else J
    keep = sort_vars({"R", "X", "Y"} | set(pa))
    M = J.sum(axis=tuple(i for i, v in enumerate(VARS) if v not in keep))
    th = bn_score.cpts["Y"][..., 1]
    tabs = []
    for r in range(bn_mass.card["R"]):
        for x in range(bn_mass.card["X"]):
            Mg = M[tuple(r if v == "R" else x if v == "X" else slice(None) for v in keep)]
            sc = np.broadcast_to(th[tuple(r if v == "R" else x if v == "X" else slice(None) for v in pa)], Mg.shape[:-1])
            tabs.append((np.ravel(sc), np.ravel(Mg[..., 1]), np.ravel(Mg[..., 0])))
    return tabs


def table_metrics(tables, thr, groups):
    """Model-implied ΔEOdds (over `groups`), balanced accuracy, TPR and FPR of the rule s >= thr[g]."""
    tp = np.array([m1[s >= thr[g]].sum() for g, (s, m1, m0) in enumerate(tables)])
    fp = np.array([m0[s >= thr[g]].sum() for g, (s, m1, m0) in enumerate(tables)])
    P = np.array([t[1].sum() for t in tables])
    N = np.array([t[2].sum() for t in tables])
    with np.errstate(divide="ignore", invalid="ignore"):
        tpr, fpr = tp / P, fp / N
    gs = np.asarray(groups, int)
    deo = max(tpr[gs].max() - tpr[gs].min(), fpr[gs].max() - fpr[gs].min()) if len(gs) >= 2 else np.nan
    return deo, 0.5 * (tp.sum() / P.sum() + 1 - fp.sum() / N.sum()), tpr, fpr


def decide(s, g, thr):
    return (s >= thr[g.astype(np.int64)]).astype(np.int8)


def gap_on(y, d, g, groups):
    pos, neg, tp, fp = group_counts(y, d, g)
    with np.errstate(divide="ignore", invalid="ignore"):
        m = _gaps(tp / pos, fp / neg, (tp + fp) / (pos + neg), np.isin(np.arange(N_GROUPS), groups))
    return m["d_eodds"], 0.5 * (tp.sum() / pos.sum() + 1 - fp.sum() / neg.sum())


def support_label(pos, neg):
    """Support class of a group in an evaluation split, from the TPR and FPR denominators."""
    m = min(pos, neg)
    if m >= CFG["min_eval"]:
        return f"primary (>={CFG['min_eval']} pos and neg)"
    if m >= CFG["min_eval_secondary"]:
        return f"secondary (>={CFG['min_eval_secondary']} pos and neg)"
    return "exploratory (<{} pos or neg)".format(CFG["min_eval_secondary"]) if pos + neg > 0 else "absent"


def per_group_rates(y, d, g, w=None):
    """Per-group denominators, selection rate, TPR and FPR of a decision rule (a rate is NaN when its denominator
    is 0). Groups absent from the split are omitted."""
    pos, neg, tp, fp = group_counts(y, d, g, w)
    cast = int if w is None else float
    out = []
    for k in range(N_GROUPS):
        n_k = pos[k] + neg[k]
        if n_k == 0:
            continue
        out.append({"group": GROUP_NAMES[k], "persons": cast(n_k), "positives": cast(pos[k]),
                    "negatives": cast(neg[k]), "selected": cast(tp[k] + fp[k]), "selection rate": (tp[k] + fp[k]) / n_k,
                    "TPR": tp[k] / pos[k] if pos[k] > 0 else np.nan, "FPR": fp[k] / neg[k] if neg[k] > 0 else np.nan,
                    "support": support_label(pos[k], neg[k])})
    return out

### 1.7 Baselines
Baselines: Kamiran-Calders reweighing (RW-S), histogram gradient boosting (GBT-S), importance-weighted EO
post-processing with a domain classifier (IW-EO, global covariate-shift correction) and target-only retraining
(LR-T, GBT-T). Settings: `CFG` (section 1.1), baselines block.

In [ ]:
def feats_hgb(data):
    a = np.stack([data[v] for v in FEATS], 1).astype(float)
    a[a < 0] = np.nan
    return a


def fit_hgb(X, y, seed, w=None, **kw):
    return HistGradientBoostingClassifier(categorical_features=[True] * X.shape[1], random_state=seed,
                                          **{**CFG["hgb_params"], **kw}).fit(X, y, sample_weight=w)


def reweighing_weights(g, y):
    """Kamiran & Calders (2012): w(g, y) = P(g) P(y) / P(g, y) on the source training split."""
    w = np.ones(len(y))
    for gg in np.unique(g):
        for yy in (0, 1):
            m = (g == gg) & (y == yy)
            if m.any():
                w[m] = (g == gg).mean() * (y == yy).mean() / m.mean()
    return w


def iw_method(S, adapt_cov, D_te, delta=None):
    """Global covariate-shift correction: domain-classifier importance weights -> weighted refit + weighted EO."""
    from scipy.sparse import vstack
    Dt = S.out.design(feats_int(adapt_cov))
    dom = LogisticRegression(C=CFG["iw_domain_C"], max_iter=5000).fit(vstack([S.Dva, Dt]),
                                                       np.r_[np.zeros(S.Dva.shape[0]), np.ones(Dt.shape[0])])
    ratio = S.Dva.shape[0] / Dt.shape[0]

    def weights(D):
        p = np.clip(dom.predict_proba(D)[:, 1], 1e-6, 1 - 1e-6)
        return p / (1 - p) * ratio

    w_tr, w_va = weights(S.Dtr_iw), weights(S.Dva)
    cap = np.quantile(w_tr, CFG["iw_weight_cap_quantile"])
    w_tr, w_va = np.minimum(w_tr, cap), np.minimum(w_va, cap)
    w_tr, w_va = w_tr / w_tr.mean(), w_va / w_va.mean()
    lr_iw = OutcomeModel(S.card).fit(S.Xtr_iw, S.ytr_iw, w=w_tr, init=S.out)
    thr, _ = select_thresholds(data_tables(lr_iw.proba(None, S.Dva), S.va["Y"], S.va["G"], w_va), S.val_groups, delta)
    return lr_iw.proba(None, D_te), thr, {"ess_frac": float(w_va.sum() ** 2 / (w_va ** 2).sum() / len(w_va))}


def _cv_ok(y, k):
    return len(y) >= 50 and np.bincount(y.astype(int), minlength=2).min() >= k


def retrain_lr(lab, card, D_te, seed, delta=None):
    y = lab["Y"].astype(int)
    if not _cv_ok(y, 5):
        return None
    X = feats_int(lab)
    folds = list(StratifiedKFold(CFG["cv_folds"], shuffle=True, random_state=seed).split(X, y))
    best = None
    for C in CFG["retrain_C_grid"]:
        oof = np.zeros(len(y))
        for a, b in folds:
            oof[b] = OutcomeModel(card, C).fit(X[a], y[a]).proba(X[b])
        o = np.clip(oof, 1e-6, 1 - 1e-6)
        ll = -np.mean(y * np.log(o) + (1 - y) * np.log(1 - o))
        if best is None or ll < best[0]:
            best = (ll, C, oof)
    _, C, oof = best
    thr, _ = select_thresholds(data_tables(oof, y, lab["G"]), support_groups(y, lab["G"], CFG["min_fit"]), delta)
    return OutcomeModel(card, C).fit(X, y).proba(None, D_te), thr, {"C": C}


def retrain_gbt(lab, te, card, seed, delta=None):
    """Gradient-boosted trees on the n target rows only. For n <= 2,000 the exact (non-histogram) implementation on
    one-hot features is much faster than HistGradientBoosting and has the same capacity here."""
    from sklearn.ensemble import GradientBoostingClassifier
    y = lab["Y"].astype(int)
    if not _cv_ok(y, 3):
        return None
    Xl, Xt = main_effects(feats_int(lab), card), main_effects(feats_int(te), card)
    make = lambda: GradientBoostingClassifier(**CFG["gbt_t_params"],
                                              min_samples_leaf=max(5, len(y) // 100), random_state=seed)
    oof = np.zeros(len(y))
    for a, b in StratifiedKFold(CFG["cv_folds"], shuffle=True, random_state=seed).split(Xl, y):
        oof[b] = make().fit(Xl[a], y[a]).predict_proba(Xl[b])[:, 1]
    thr, _ = select_thresholds(data_tables(oof, y, lab["G"]), support_groups(y, lab["G"], CFG["min_fit"]), delta)
    return make().fit(Xl, y).predict_proba(Xt)[:, 1], thr, {}

### 1.8 BN-LFA (proposed method)
BN-LFA (proposed): localized, fairness-aware factor adaptation.

1. Shift evidence per factor: Bayes factor of "this CPT shifted" vs "unchanged" from the n labelled (and optional
   unlabelled) target rows, by Bayesian model averaging over structured shift families (composition CPTs: global tilt,
   tilt per parent level, per-configuration Dirichlet; outcome CPD: logit residuals with intercept / one feature /
   all main effects, Laplace evidence).
2. Decision impact per factor: exact change of the source fairness rule's model-implied ΔEOdds and balanced accuracy
   when only that factor is replaced by its update (exact enumeration of the joint).
3. Selection: at most two factors with log BF > 0, ranked by P(shift) x (|Δ ΔEOdds| + |Δ BA|).
4. Rule: reweight the source validation split by the density ratio of the selected factors only and refit the
   equalized-odds group thresholds within the utility tolerance δ. With nothing selected the rule is EO-S.

In [ ]:
def dirichlet_logml(prior, counts, alpha):
    a = alpha * prior
    return float((gammaln(alpha) - gammaln(alpha + counts.sum(1))).sum() + (gammaln(a + counts) - gammaln(a)).sum())


def tilt_fit(theta, counts, sigma=None, iters=60):
    """MAP of theta_t(k|u) ∝ theta(k|u) exp(beta_k) (beta_0 = 0, beta ~ N(0, sigma^2 I)) and its Laplace log-evidence."""
    sigma = CFG["tilt_sigma"] if sigma is None else sigma
    K = theta.shape[1]
    d, logt, n_u = K - 1, np.log(theta), counts.sum(1)
    beta = np.zeros(K)

    def state(b):
        z = logt + b
        z = z - z.max(1, keepdims=True)
        q = np.exp(z)
        q /= q.sum(1, keepdims=True)
        A = q * n_u[:, None]
        return q, (np.diag(A.sum(0)) - A.T @ q)[1:, 1:] + np.eye(d) / sigma ** 2

    for _ in range(iters):
        q, hpost = state(beta)
        step = np.linalg.solve(hpost, (counts - n_u[:, None] * q).sum(0)[1:] - beta[1:] / sigma ** 2)
        beta[1:] += step
        if np.abs(step).max() < 1e-10:
            break
    q, hpost = state(beta)
    ll = float((counts * np.log(q)).sum())
    return q, beta, ll - (beta[1:] ** 2).sum() / (2 * sigma ** 2) - d * np.log(sigma) - 0.5 * np.linalg.slogdet(hpost)[1]


def factor_shift(theta_s, counts, update="tilt+dir", parent_names=()):
    """Shift evidence (BMA over shift families) and posterior-mean update for a composition CPT.
    update="dir" is the per-configuration Dirichlet ablation."""
    ALPHAS = np.array(CFG["alpha_grid"], float)
    K = theta_s.shape[-1]
    th, c = theta_s.reshape(-1, K), counts.reshape(-1, K)
    e0 = float((c * np.log(th)).sum())
    ml_dir = np.array([dirichlet_logml(th, c, a) for a in ALPHAS])
    e_dir = logsumexp(ml_dir) - np.log(len(ALPHAS))
    if update == "dir":
        a_star = ALPHAS[int(np.argmax(ml_dir))]
        new = (c + a_star * th) / (c.sum(1, keepdims=True) + a_star)
        return {"theta": new.reshape(theta_s.shape), "log_bf": float(e_dir - e0), "shift_size": np.nan,
                "alpha": float(a_star), "n": float(c.sum()), "update": "dirichlet"}
    q, beta, e = tilt_fit(th, c)
    fams = {"global tilt": (e, q, np.abs(beta).max())}
    pshape = theta_s.shape[:-1]
    if pshape:
        lev = np.indices(pshape).reshape(len(pshape), -1)
        for j in range(len(pshape)):
            qj, ej, bj = np.empty_like(th), 0.0, 0.0
            for l in range(pshape[j]):
                rows = lev[j] == l
                q_l, b_l, e_l = tilt_fit(th[rows], c[rows])
                qj[rows], ej, bj = q_l, ej + e_l, max(bj, np.abs(b_l).max())
            name = parent_names[j] if j < len(parent_names) else str(j)
            fams[f"tilt by {name}"] = (ej, qj, bj)
    evid = [f[0] for f in fams.values()] + [e_dir]
    best = max(fams, key=lambda k: fams[k][0])
    base = fams[best][1]
    a_star = ALPHAS[int(np.argmax([dirichlet_logml(base, c, a) for a in ALPHAS]))]
    new = (c + a_star * base) / (c.sum(1, keepdims=True) + a_star)
    return {"theta": new.reshape(theta_s.shape), "log_bf": float(logsumexp(evid) - np.log(len(evid)) - e0),
            "shift_size": float(fams[best][2]), "alpha": float(a_star), "n": float(c.sum()), "update": best}


def glm_residual(off, Phi, y, sig, iters=50):
    """MAP of a logit-scale residual delta (prior N(0, diag(sig^2))) with fixed offset, and its Laplace log-evidence."""
    s2 = np.asarray(sig, float) ** 2
    d = np.zeros(Phi.shape[1])
    for _ in range(iters):
        p = expit(off + Phi @ d)
        H = (Phi * (p * (1 - p))[:, None]).T @ Phi + np.diag(1 / s2)
        step = np.linalg.solve(H, Phi.T @ (y - p) - d / s2)
        d += step
        if np.abs(step).max() < 1e-9:
            break
    eta = off + Phi @ d
    p = expit(eta)
    H = (Phi * (p * (1 - p))[:, None]).T @ Phi + np.diag(1 / s2)
    ll = float((y * eta - np.logaddexp(0, eta)).sum())
    return d, ll - 0.5 * (d ** 2 / s2).sum() - 0.5 * np.log(s2).sum() - 0.5 * np.linalg.slogdet(H)[1]


def outcome_shift(bn_s, adapt, update="structured"):
    """Shift evidence and update for the log-linear outcome CPD (labelled rows only)."""
    th = bn_s.cpts["Y"]
    if update == "dir":
        return factor_shift(th, family_counts(adapt, "Y", bn_s.parents["Y"], bn_s.card), "dir")
    X = feats_int(adapt)
    y = adapt["Y"].astype(int)
    ok = (y >= 0) & np.all(X >= 0, axis=1)
    X, y = X[ok], y[ok].astype(float)
    if len(y) == 0:
        return {"theta": th, "log_bf": 0.0, "shift_size": 0.0, "alpha": np.nan, "n": 0.0, "update": "none"}
    p_s = th[tuple(X[:, j] for j in range(len(FEATS)))][:, 1]
    off = np.log(p_s) - np.log1p(-p_s)
    Phi = main_effects(X, bn_s.card)
    e0 = float((y * off - np.logaddexp(0, off)).sum())
    starts = np.cumsum([1] + [bn_s.card[v] for v in FEATS])
    families = {"intercept": [0], **{f"intercept + {v}": [0] + list(range(starts[i], starts[i + 1]))
                                     for i, v in enumerate(FEATS)}, "all main effects": list(range(Phi.shape[1]))}
    fits = {}
    for name, cols in families.items():
        sig = np.r_[1.0, np.full(len(cols) - 1, CFG["glm_sigma"])]
        fits[name] = glm_residual(off, Phi[:, cols], y, sig)
    best = max(fits, key=lambda k: fits[k][1])
    delta_hat = fits[best][0]
    # tabulate the residual over every parent configuration: one-hot main effects add one lookup per feature
    X_all = all_configs(bn_s.card)
    lg = np.log(th[..., 1]).reshape(-1) - np.log(th[..., 0]).reshape(-1) + delta_hat[0]
    for i, v in enumerate(FEATS):
        eff = np.zeros(bn_s.card[v])
        for c, dv in zip(families[best][1:], delta_hat[1:]):
            if starts[i] <= c < starts[i + 1]:
                eff[c - starts[i]] = dv
        if eff.any():
            lg = lg + eff[X_all[:, i]]
    p = expit(lg).reshape(th.shape[:-1])
    assert np.allclose(p[tuple(X[:, j] for j in range(len(FEATS)))], expit(off + Phi[:, families[best]] @ delta_hat),
                       atol=1e-9), "tabulated outcome update disagrees with the fitted residual"
    evid = [f[1] for f in fits.values()]
    return {"theta": np.stack([1 - p, p], -1), "log_bf": float(logsumexp(evid) - np.log(len(evid)) - e0),
            "shift_size": float(np.abs(delta_hat).max()), "alpha": np.nan, "n": float(len(y)), "update": best}


def update_factor(bn_s, v, adapt, update="structured"):
    if v == "Y":
        return outcome_shift(bn_s, adapt, update)
    return factor_shift(bn_s.cpts[v], family_counts(adapt, v, bn_s.parents[v], bn_s.card),
                        "tilt+dir" if update == "structured" else "dir", parent_names=bn_s.parents[v])


def factor_diagnostic(bn_s, rule_thr, adapt, groups, update="structured"):
    """Bayes-factor evidence + exact decision impact for every factor. Returns (table, updated factors)."""
    base_deo, base_ba, _, _ = table_metrics(model_tables(bn_s), rule_thr, groups)
    rows, new = [], {}
    for v in VARS:
        res = update_factor(bn_s, v, adapt, update)
        new[v] = res["theta"]
        deo_v, ba_v, _, _ = table_metrics(model_tables(bn_s.copy(**{v: res["theta"]}), bn_s), rule_thr, groups)
        rows.append({"factor": v, "family": f"P({v}|{','.join(bn_s.parents[v])})", "n_obs": res["n"],
                     "log_bf": res["log_bf"], "fair_impact": deo_v - base_deo, "util_impact": ba_v - base_ba,
                     "update": res["update"], "alpha": res["alpha"], "shift_size": res["shift_size"]})
    df = pd.DataFrame(rows)
    df["p_shift"] = expit(df["log_bf"].clip(-700, 700))
    df["impact"] = df["p_shift"] * (df["fair_impact"].abs() + df["util_impact"].abs())
    return df, new


def select_factors(diag, rule, rng=None, k=None):
    k = CFG["max_factors"] if k is None else k
    if rule == "all":
        return list(VARS)
    if rule == "rand":
        return sorted(rng.choice(VARS, size=k, replace=False).tolist(), key=VARS.index)
    cand = diag[diag["log_bf"] > 0]
    return cand.sort_values("impact" if rule == "lfa" else "log_bf", ascending=False)["factor"].head(k).tolist()


def factor_ratio_rule(S, F, upd, te, s_te_lr, delta=None):
    """Proposed estimator: reweight the source validation split by the density ratio of the updated factors only,
    w(x, y) = prod_{v in F} theta_t(v | pa) / theta_s(v | pa), and select EO thresholds on the reweighted split.
    Every factor that is not updated cancels from the ratio, so its misspecification cannot bias the estimate."""
    w = np.ones(len(S.va["Y"]))
    for v in F:
        idx = tuple(S.va[u].astype(np.int64) for u in S.bn.scope(v))
        w = w * upd[v][idx] / S.bn.cpts[v][idx]
    cap = np.quantile(w, CFG["lfa_weight_cap_quantile"])
    w = np.minimum(w, cap)
    w = w / w.mean()
    if "Y" in F:                                         # outcome factor updated: use the adapted outcome CPD as score
        bn_y = S.bn.copy(Y=upd["Y"])
        s_va, s_te = bn_y.predict_proba(S.va), bn_y.predict_proba(te)
    else:
        s_va, s_te = S.s_va_lr, s_te_lr
    thr, _ = select_thresholds(data_tables(s_va, S.va["Y"], S.va["G"], w), S.val_groups, delta)
    return s_te, thr, {"ess_frac": float(w.sum() ** 2 / (w ** 2).sum() / len(w))}


def table_loglik(table, data, scope):
    """log table[x_scope] for every row whose family is fully observed (NaN otherwise)."""
    arrs = [data[u].astype(np.int64) for u in scope]
    ok = np.all([a >= 0 for a in arrs], axis=0)
    out = np.full(len(arrs[0]), np.nan)
    out[ok] = np.log(table[tuple(a[ok] for a in arrs)])
    return out


def heldout_factor_loglik(bn_s, new, te, w=None):
    """Mean held-out log-likelihood per person of every factor on the target test split, under the source CPT and
    under its update from the adaptation sample (probabilistic quality of the adapted network, factor by factor).
    The joint log-likelihood of any adapted network is the sum of its factors' terms."""
    src, upd = [], []
    for v in VARS:
        scope = bn_s.scope(v)
        ll_s, ll_u = table_loglik(bn_s.cpts[v], te, scope), table_loglik(new[v], te, scope)
        ok = ~np.isnan(ll_s)
        ww = np.ones(len(ll_s))[ok] if w is None else np.asarray(w, float)[ok]
        src.append(float((ww * ll_s[ok]).sum() / ww.sum()))
        upd.append(float((ww * ll_u[ok]).sum() / ww.sum()))
    return {"test_ll_source": src, "test_ll_updated": upd}

### 1.9 One adaptation run (every method)
One fitted source domain (all source-side models and fairness rules) and one adaptation run that evaluates every
method on a target test split.

Information available to every adaptation method in a run: the fitted source models, the source validation split,
and the same n labelled target rows (plus, in +U variants, the same unlabelled target covariates). Target test rows
are used only for evaluation.

In [ ]:
ALL_METHODS = ["LR-S", "GBT-S", "RW-S", "EO-S", "TT", "IW-EO", "LR-T", "GBT-T", "BN-S", "BN-all", "BN-LFA",
               "BN-LL", "BN-rand", "BN-LFA-dir", "BN-LFA-model", "BN-LFA-emp"]
METHOD_FAMILY = {"LR-S": "predictive (no mitigation)", "GBT-S": "predictive (no mitigation)",
                 "RW-S": "fairness baseline", "EO-S": "fairness baseline", "TT": "fairness baseline",
                 "IW-EO": "global adaptation", "LR-T": "full retraining", "GBT-T": "full retraining",
                 "BN-S": "graphical-model baseline", "BN-all": "graphical-model baseline", "BN-LFA": "proposed",
                 "BN-LL": "ablation", "BN-rand": "ablation", "BN-LFA-dir": "ablation", "BN-LFA-model": "ablation",
                 "BN-LFA-emp": "ablation"}
# Every score model takes race R and sex X as inputs; methods with group thresholds also use the group at decision time.
USES_PROTECTED = {m: ("R, X as features; single threshold" if m in ("LR-S", "GBT-S", "RW-S")
                      else "R, X as features; group-specific thresholds") for m in ALL_METHODS}
FAILED = []                         # method-runs that could not be fitted, with the reason (written to the run log)


class Source:
    """Everything fitted on one source domain (train = P1, validation = P2)."""

    def __init__(self, tag, train, val, card, parents=None, bn=None, outcome=None, gbt=True, rw=True, seed=0):
        self.tag, self.tr, self.va, self.card = tag, train, val, card
        if bn is None:
            parents = parents if parents is not None else learn_structure(train, card, val=val)[0]
            bn, outcome = fit_bn(train, parents, card, outcome)
        self.bn, self.out = bn, outcome                      # self.out is LR-S (the BN's outcome CPD)
        self.val_groups = support_groups(val["Y"], val["G"], CFG["min_fit"])
        self.thr_bn_eo, self.info_bn = select_thresholds(model_tables(bn), self.val_groups)
        Xtr = feats_int(train)
        self.Dva = self.out.design(feats_int(val))
        tv = lambda s, w=None: data_tables(s, val["Y"], val["G"], w)
        self.s_va_lr = self.out.proba(None, self.Dva)
        self.thr_lr_eo, _ = select_thresholds(tv(self.s_va_lr), self.val_groups)
        self.thr_lr_1, _ = select_thresholds(tv(self.s_va_lr), self.val_groups, mode="single")
        self.rw = None
        if rw:
            self.rw = OutcomeModel(card).fit(Xtr, train["Y"], w=reweighing_weights(train["G"], train["Y"]))
            self.thr_rw_1, _ = select_thresholds(tv(self.rw.proba(None, self.Dva)), self.val_groups, mode="single")
        self.gbt = None
        if gbt:
            self.gbt = fit_hgb(feats_hgb(train), train["Y"], seed)
            s = self.gbt.predict_proba(feats_hgb(val))[:, 1]
            self.thr_gbt_1, _ = select_thresholds(tv(s), self.val_groups, mode="single")
            self.thr_gbt_eo, _ = select_thresholds(tv(s), self.val_groups)
        rng = np.random.default_rng(stable_seed("iw", tag))
        idx = rng.choice(len(train["Y"]), min(CFG["iw_source_rows"], len(train["Y"])), replace=False)
        self.Xtr_iw, self.ytr_iw = Xtr[idx], train["Y"][idx]
        self.Dtr_iw = self.out.design(self.Xtr_iw)
        self._cache = {}

    def target_cache(self, key, te):
        """Scores of the fixed source models on a target test set (computed once per target)."""
        if key not in self._cache:
            X = feats_int(te)
            D = self.out.design(X)
            c = {"X": X, "D": D, "lr": self.out.proba(None, D), "bn": self.bn.predict_proba(te)}
            if self.rw is not None:
                c["rw"] = self.rw.proba(None, D)
            if self.gbt is not None:
                c["gbt"] = self.gbt.predict_proba(feats_hgb(te))[:, 1]
            self._cache[key] = c
        return self._cache[key]


def run_one(S, tgt_tag, n, seed, doms, unlabeled=0, methods=None, store=None, delta=None,
            exclude_factors=(), exact=None):
    """One adaptation run: draw the labelled budget from the target pool, fit every method, evaluate on the target test."""
    delta = CFG["delta_ba"] if delta is None else delta
    methods = methods or ALL_METHODS
    T = doms[tgt_tag]
    te = T["test"]
    c = S.target_cache(tgt_tag, te)
    rng = np.random.default_rng(stable_seed("adapt", S.tag, tgt_tag, seed))
    perm = rng.permutation(T["idx_pool"])                        # nested budgets: first n of a seed-specific permutation
    lab = take(T["all"], perm[:n])
    unl = take(T["all"], perm[n:n + unlabeled])
    unl["Y"] = np.full(len(unl["Y"]), -1, np.int8)
    adapt = concat(lab, unl) if unlabeled else lab
    out, info, diag = {}, {}, None
    thr_eo, thr_bn = S.thr_lr_eo, S.thr_bn_eo
    if delta != CFG["delta_ba"]:                                 # trade-off curves: re-select the source rules too
        thr_eo = select_thresholds(data_tables(S.s_va_lr, S.va["Y"], S.va["G"]), S.val_groups, delta)[0]
        thr_bn = select_thresholds(model_tables(S.bn), S.val_groups, delta)[0]
    out["LR-S"], out["EO-S"] = (c["lr"], S.thr_lr_1), (c["lr"], thr_eo)
    if S.rw is not None:
        out["RW-S"] = (c["rw"], S.thr_rw_1)
    if S.gbt is not None:
        out["GBT-S"] = (c["gbt"], S.thr_gbt_1)
    out["BN-S"] = (c["bn"], thr_bn)
    bn_family = ("BN-all", "BN-LFA", "BN-LL", "BN-rand", "BN-LFA-dir", "BN-LFA-model", "BN-LFA-emp")
    if n == 0 and unlabeled == 0:                                # zero-label transfer: nothing can be updated
        out["TT"] = out["IW-EO"] = out["EO-S"]
        for m in bn_family:
            out[m] = out["BN-S"] if m == "BN-LFA-model" else out["EO-S"]
    else:
        if n > 0 and "TT" in methods:                            # groups without target support keep the EO-S threshold
            s_lab = S.out.proba(feats_int(lab))
            out["TT"] = (c["lr"], select_thresholds(data_tables(s_lab, lab["Y"], lab["G"]),
                                                    support_groups(lab["Y"], lab["G"], CFG["min_fit"]), delta,
                                                    fallback=thr_eo)[0])
        if "IW-EO" in methods:
            s, thr, inf = iw_method(S, adapt, c["D"], delta)
            out["IW-EO"], info["IW-EO"] = (s, thr), inf
        if n > 0 and "LR-T" in methods:
            r = retrain_lr(lab, S.card, c["D"], seed, delta)
            if r is not None:
                out["LR-T"] = r[:2]
            else:
                FAILED.append({"source": S.tag, "target": tgt_tag, "n": n, "seed": seed, "method": "LR-T",
                               "reason": "skipped: fewer than 50 labels or fewer than 5 of a class"})
        if n > 0 and "GBT-T" in methods:
            r = retrain_gbt(lab, te, S.card, seed, delta)
            if r is not None:
                out["GBT-T"] = r[:2]
            else:
                FAILED.append({"source": S.tag, "target": tgt_tag, "n": n, "seed": seed, "method": "GBT-T",
                               "reason": "skipped: fewer than 50 labels or fewer than 3 of a class"})
        if set(bn_family) & set(methods):
            diag, new = factor_diagnostic(S.bn, thr_eo, adapt, S.val_groups)
            diag = diag.assign(**heldout_factor_loglik(S.bn, new, te, None if exact is None else exact[0]))
            if exclude_factors:
                diag = diag[~diag.factor.isin(exclude_factors)]
            F_lfa = select_factors(diag, "lfa")
            for name, rule in (("BN-LFA", "lfa"), ("BN-LL", "ll"), ("BN-rand", "rand"), ("BN-all", "all")):
                if name not in methods and name != "BN-LFA":
                    continue
                F = F_lfa if rule == "lfa" else select_factors(diag, rule, rng)
                if F:
                    s, thr, inf = factor_ratio_rule(S, F, new, te, c["lr"], delta)
                    out[name] = (s, thr)
                    info[name] = {"factors": "+".join(F), **inf}
                else:
                    out[name], info[name] = out["EO-S"], {"factors": "-", "ess_frac": 1.0}
            if "BN-LFA-dir" in methods:
                if F_lfa:
                    upd = {v: update_factor(S.bn, v, adapt, "dir")["theta"] for v in F_lfa}
                    out["BN-LFA-dir"] = factor_ratio_rule(S, F_lfa, upd, te, c["lr"], delta)[:2]
                else:
                    out["BN-LFA-dir"] = out["EO-S"]
            if "BN-LFA-model" in methods:
                bn_a = S.bn.copy(**{v: new[v] for v in F_lfa})
                out["BN-LFA-model"] = (bn_a.predict_proba(te),
                                       select_thresholds(model_tables(bn_a), S.val_groups, delta)[0])
            if "BN-LFA-emp" in methods and n > 0:
                s_lab = S.bn.copy(Y=new["Y"]).predict_proba(lab) if "Y" in F_lfa else S.out.proba(feats_int(lab))
                thr = select_thresholds(data_tables(s_lab, lab["Y"], lab["G"]),
                                        support_groups(lab["Y"], lab["G"], CFG["min_fit"]), delta,
                                        fallback=out["BN-LFA"][1])[0]
                out["BN-LFA-emp"] = (out["BN-LFA"][0], thr)
    rows, cache = [], {}
    for m in methods:
        if m not in out:
            continue
        s, thr = out[m]
        d = decide(s, te["G"], thr)
        if id(s) not in cache:
            cache[id(s)] = met = evaluate(te["Y"], s, d, te["G"], exact=exact)
        else:
            met = {**cache[id(s)], **evaluate(te["Y"], s, d, te["G"], scores_metrics=False, exact=exact)}
        rows.append({"source": S.tag, "target": tgt_tag, "pair": f"{S.tag}→{tgt_tag}", "n": n, "seed": seed,
                     "unlabeled": unlabeled, "method": m, **met, **info.get(m, {})})
        if store is not None:
            store[(S.tag, tgt_tag, n, seed, m)] = np.packbits(d)
    return rows, diag

### 1.10 Controlled simulation (synthetic)
Experiment 4 (SYNTHETIC; every number from here is simulated): a known target network obtained by shifting chosen
factors of the CA18 network, exact evaluation over every configuration, and recovery of the shifted factors.

Scenarios: null (no shift), demographic (P(R) replaced by TX18's race shares), composition (education shifted
downwards for Black and Other-race persons), mechanism (outcome logit lowered for low education, raised otherwise),
mixed (composition + mechanism) and unselected (mixed + an hours shift for women: three shifted factors while at most
two may be updated). Magnitudes: `CFG` (section 1.1), simulation block.

In [ ]:
SIM_SCENARIOS = ["null", "demographic", "composition", "mechanism", "mixed", "unselected"]
SIM_METHODS = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA", "BN-LL", "BN-LFA-model"]
SIM_U_METHODS = ["IW-EO", "BN-LFA"]


def _tilt_rows(cpt, beta, row_mask):
    """Multiply the selected rows of a CPT by exp(beta_k) and renormalise."""
    K = cpt.shape[-1]
    c = cpt.reshape(-1, K).copy()
    t = c[row_mask] * np.exp(beta)[None, :]
    c[row_mask] = t / t.sum(1, keepdims=True)
    return c.reshape(cpt.shape)


def _rows_where(parents, card, cond):
    """Boolean mask over parent configurations (C order) where cond(dict parent->value) holds."""
    grids = np.indices([card[p] for p in parents]).reshape(len(parents), -1) if parents else np.zeros((0, 1), int)
    return np.array([cond(dict(zip(parents, col))) for col in grids.T])


def make_target(base, scenario, card, ref_train):
    """Known target network: `base` with the scenario's factors shifted. Returns (network, shifted factors)."""
    pa, new, shifted = base.parents, {}, []
    if scenario in ("demographic",):
        p = np.bincount(ref_train["R"], minlength=card["R"]) + 0.5
        new["R"] = p / p.sum()
        shifted.append("R")
    if scenario in ("composition", "mixed", "unselected"):
        K = card["E"]
        beta = CFG["sim_composition_tilt"] * np.arange(K)                 # shift education toward lower levels
        if "R" in pa["E"]:
            races = CFG["sim_composition_races"]
            mask = _rows_where(pa["E"], card, lambda d: d["R"] in races)
        else:
            mask = np.ones(int(np.prod([card[p] for p in pa["E"]])), bool)
        new["E"] = _tilt_rows(base.cpts["E"], beta, mask)
        shifted.append("E")
    if scenario in ("mechanism", "mixed", "unselected"):
        z = next(p for p in ("E", "O", "H", "C") if p in pa["Y"])          # a non-protected parent of Y
        Kz = card[z]
        lo = set(range(Kz // 2)) if z != "O" else {4, 5, 6, 7}             # lower-paying half of the parent's levels
        th = base.cpts["Y"].reshape(-1, 2).copy()
        grids = np.indices([card[p] for p in pa["Y"]]).reshape(len(pa["Y"]), -1).T
        zi = pa["Y"].index(z)
        delta = np.where(np.isin(grids[:, zi], list(lo)), CFG["sim_mechanism_low"], CFG["sim_mechanism_high"])
        lg = np.log(th[:, 1] / th[:, 0]) + delta
        th[:, 1] = expit(lg)
        th[:, 0] = 1 - th[:, 1]
        new["Y"] = th.reshape(base.cpts["Y"].shape)
        shifted.append("Y")
    if scenario == "unselected":
        Kh = card["H"]
        mask = _rows_where(pa["H"], card, lambda d: d.get("X", 1) == 1)
        new["H"] = _tilt_rows(base.cpts["H"], CFG["sim_unselected_tilt"] * np.arange(Kh), mask)
        shifted.append("H")
    return base.copy(**new), shifted


def exact_test_set(bn_true, min_mass=None):
    """All configurations of (R,X,C,E,W,O,H,Y) with weights P_true(config): an exact, noise-free 'test set'."""
    min_mass = CFG["sim_min_mass"] if min_mass is None else min_mass
    J = bn_true.joint()
    idx = np.indices(J.shape).reshape(len(VARS), -1)
    te = {v: idx[i].astype(np.int8) for i, v in enumerate(VARS)}
    te["G"] = (2 * te["R"].astype(int) + te["X"]).astype(np.int8)
    w = J.reshape(-1)
    pos = np.bincount(te["G"][te["Y"] == 1], w[te["Y"] == 1], N_GROUPS)
    neg = np.bincount(te["G"][te["Y"] == 0], w[te["Y"] == 0], N_GROUPS)
    return te, w, (pos >= min_mass) & (neg >= min_mass)


def run_simulation(base, card, ref_train):
    """All scenarios x replications x budgets; exact target metrics. Returns (method runs, factor-recovery rows)."""
    sim_rows, sim_diag_rows = [], []
    t0 = time.time()
    for scen in SIM_SCENARIOS:
        bn_t, shifted = make_target(base, scen, card, ref_train)
        te_exact, w_exact, support = exact_test_set(bn_t)
        exact = (w_exact, support)
        for rep in range(CFG["sim_reps"]):
            rng = np.random.default_rng(stable_seed("sim", scen, rep))
            sz = CFG["sim_sizes"]
            src_tr, src_va = base.sample(sz["source_train"], rng), base.sample(sz["source_val"], rng)
            pool = bn_t.sample(sz["target_pool"], rng)
            S = Source(f"sim-{scen}-{rep}", src_tr, src_va, card, parents=base.parents, gbt=False, rw=False)
            dom = {"sim": {"all": pool, "idx_pool": np.arange(len(pool["Y"])), "test": te_exact}}
            # the source rule's true target gap (known) and its model-implied estimate after adaptation
            true_src = table_metrics(model_tables(bn_t, S.bn), S.thr_lr_eo, np.where(support)[0])[0]
            perm = np.random.default_rng(stable_seed("adapt", S.tag, "sim", rep)).permutation(len(pool["Y"]))
            for n in CFG["sim_budgets"]:
                rows, dg = run_one(S, "sim", n, rep, dom, methods=SIM_METHODS, exact=exact)
                rows_u, dg_u = run_one(S, "sim", n, rep, dom, methods=SIM_U_METHODS, unlabeled=CFG["unlabeled_pool"],
                                       exact=exact)
                for r in rows_u:
                    r["method"] += " +U"
                for r in rows + rows_u:
                    r.update({"scenario": scen, "rep": rep, "shifted": "+".join(shifted)})
                sim_rows.extend(rows + rows_u)
                for rule, dgx, lbl in (("lfa", dg, "LFA"), ("ll", dg, "LL"), ("lfa", dg_u, "LFA +U")):
                    F = set(select_factors(dgx, rule))
                    tru = set(shifted)
                    sim_diag_rows.append({"scenario": scen, "rep": rep, "n": n, "selector": lbl,
                                          "selected": "+".join(sorted(F, key=VARS.index)) or "-",
                                          "recall": len(F & tru) / len(tru) if tru else np.nan,
                                          "false selections": len(F - tru), "exact match": F == tru})
                new = factor_diagnostic(S.bn, S.thr_lr_eo, take(pool, perm[:n]), S.val_groups)[1]
                for lbl, F in (("BN-all", VARS), ("BN-LFA", select_factors(dg, "lfa"))):
                    bn_hat = S.bn.copy(**{v: new[v] for v in F})
                    est = table_metrics(model_tables(bn_hat, S.bn), S.thr_lr_eo, np.where(support)[0])[0]
                    sim_rows.append({"scenario": scen, "rep": rep, "n": n, "method": f"estimate[{lbl}]",
                                     "d_eodds": est, "true_source_rule_d_eodds": true_src})
        print(f"  scenario {scen:12s} done ({time.time() - t0:.0f}s)")
    return pd.DataFrame(sim_rows), pd.DataFrame(sim_diag_rows)

### 1.11 Uncertainty: household bootstrap, Holm, replicate weights
Uncertainty and multiplicity.

* Paired household (cluster) bootstrap of each target test split with every method's decisions held fixed: the same
  resampled households are used for all methods, so differences are paired. Metrics are averaged over the five
  adaptation seeds inside each replicate and over the nine transfer pairs (fixed-model evaluation uncertainty;
  seed-to-seed spread is reported separately as training/adaptation variability).
* Holm correction over the preregistered confirmatory family (BN-LFA vs BN-all, TT, IW-EO at n = 200 and 1,000).
* Survey-weighted estimates with the ACS successive-difference replicate weights: SE = sqrt(4/80 sum_r (θ_r - θ)^2).

In [ ]:
BOOT_COMPARE = ["BN-LFA", "BN-all", "TT", "IW-EO", "BN-S", "EO-S"]
CONFIRMATORY = ("BN-all", "TT", "IW-EO")
EXPLORATORY = ("EO-S", "BN-S")
SW_METHODS = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA"]


def household_aggregates(te, d):
    """Per-household counts of (TP, FN, FP, TN) for each of the 18 groups: array (H, 18*4)."""
    cat = np.where(te["Y"] == 1, np.where(d == 1, 0, 1), np.where(d == 1, 2, 3))
    col = te["G"].astype(np.int64) * 4 + cat
    hh = te["_hh"]
    return np.bincount(hh * (N_GROUPS * 4) + col, minlength=te["_H"] * N_GROUPS * 4).reshape(te["_H"], N_GROUPS * 4)


def metrics_from_counts(cnt, support):
    c = cnt.reshape(cnt.shape[0], N_GROUPS, 4)
    tp, fn, fp, tn = c[..., 0], c[..., 1], c[..., 2], c[..., 3]
    with np.errstate(divide="ignore", invalid="ignore"):
        tpr, fpr = tp / (tp + fn), fp / (fp + tn)
    s = support
    deo = np.maximum(np.nanmax(tpr[:, s], 1) - np.nanmin(tpr[:, s], 1), np.nanmax(fpr[:, s], 1) - np.nanmin(fpr[:, s], 1))
    ba = 0.5 * (tp.sum(1) / (tp + fn).sum(1) + tn.sum(1) / (fp + tn).sum(1))
    return deo, ba


def boot_p(diff):
    return float(min(1.0, 2 * min(((diff <= 0).sum() + 1) / (len(diff) + 1), ((diff >= 0).sum() + 1) / (len(diff) + 1))))


def holm(p):
    """Holm step-down adjusted p-values."""
    p = np.asarray(p, float)
    order_p = np.argsort(p)
    out = np.zeros(len(p))
    running = 0.0
    for rank, i in enumerate(order_p):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        out[i] = running
    return out


def household_bootstrap(doms, store):
    """Seed-averaged, pair-averaged ΔEOdds and BA under B household resamples of every target test split."""
    B = CFG["n_boot"]
    stat = {(m, n): {"deo": np.zeros(B), "ba": np.zeros(B), "deo0": 0.0, "ba0": 0.0} for m in BOOT_COMPARE
            for n in CFG["confirmatory_budgets"]}
    pair_boot = {}
    pairs = CFG["pairs"]
    for t_tag in sorted({t for _, t in pairs}, key=CFG["domains"].index):
        te = dict(doms[t_tag]["test"])
        uniq, inv = np.unique(te["HH"], return_inverse=True)
        te["_hh"], te["_H"] = inv.ravel(), len(uniq)
        upos = np.bincount(te["G"][te["Y"] == 1].astype(int), minlength=N_GROUPS)
        uneg = np.bincount(te["G"][te["Y"] == 0].astype(int), minlength=N_GROUPS)
        support = (upos >= CFG["min_eval"]) & (uneg >= CFG["min_eval"])
        rng = np.random.default_rng(stable_seed("bootstrap", t_tag))
        Wm = np.zeros((B, te["_H"]), np.float32)
        for b in range(B):
            Wm[b] = np.bincount(rng.integers(0, te["_H"], te["_H"]), minlength=te["_H"])
        for s_tag in [s for s, t in pairs if t == t_tag]:
            for n in CFG["confirmatory_budgets"]:
                for m in BOOT_COMPARE:
                    deo_b, ba_b, deo_0, ba_0 = np.zeros(B), np.zeros(B), [], []
                    for seed in CFG["seeds"]:
                        key = (s_tag, t_tag, n, seed, m)
                        d = np.unpackbits(store[key], count=len(te["Y"])).astype(np.int8)
                        A = household_aggregates(te, d).astype(np.float32)
                        db, bb = metrics_from_counts(Wm @ A, support)
                        d0, b0 = metrics_from_counts(A.sum(0, keepdims=True), support)
                        deo_b += db / len(CFG["seeds"])
                        ba_b += bb / len(CFG["seeds"])
                        deo_0.append(d0[0])
                        ba_0.append(b0[0])
                    st = stat[(m, n)]
                    st["deo"] += deo_b / len(pairs)
                    st["ba"] += ba_b / len(pairs)
                    st["deo0"] += np.mean(deo_0) / len(pairs)
                    st["ba0"] += np.mean(ba_0) / len(pairs)
                    pair_boot[(f"{s_tag}→{t_tag}", n, m)] = (np.mean(deo_0), deo_b)
        del Wm
    return stat, pair_boot


def confirmatory_table(stat):
    """Preregistered family: BN-LFA minus each comparator at each confirmatory budget; Holm-adjusted p-values;
    balanced-accuracy non-inferiority at margin δ."""
    conf_rows = []
    for n in CFG["confirmatory_budgets"]:
        for comp in CONFIRMATORY:
            a, b_ = stat[("BN-LFA", n)], stat[(comp, n)]
            diff, dba = a["deo"] - b_["deo"], a["ba"] - b_["ba"]
            conf_rows.append({"n": n, "comparison": f"BN-LFA − {comp}", "Δ mean ΔEOdds": a["deo0"] - b_["deo0"],
                              "95% CI low": np.percentile(diff, 2.5), "95% CI high": np.percentile(diff, 97.5),
                              "p (bootstrap)": boot_p(diff), "Δ BA": a["ba0"] - b_["ba0"],
                              "BA 95% CI low": np.percentile(dba, 2.5), "BA 95% CI high": np.percentile(dba, 97.5)})
    conf = pd.DataFrame(conf_rows)
    conf["p (Holm)"] = holm(conf["p (bootstrap)"].to_numpy())
    conf["fairness claim (Δ < 0, Holm α = 0.05)"] = (conf["p (Holm)"] < CFG["holm_alpha"]) & (conf["Δ mean ΔEOdds"] < 0)
    conf["BA non-inferior (CI low > −0.02)"] = conf["BA 95% CI low"] > -CFG["delta_ba"]
    return conf


def exploratory_table(stat):
    """Not part of the preregistered family (unadjusted): adaptation vs. no adaptation."""
    rows = []
    for n in CFG["confirmatory_budgets"]:
        for comp in EXPLORATORY:
            a, b_ = stat[("BN-LFA", n)], stat[(comp, n)]
            diff = a["deo"] - b_["deo"]
            rows.append({"n": n, "comparison (exploratory)": f"BN-LFA − {comp}", "Δ mean ΔEOdds": a["deo0"] - b_["deo0"],
                         "95% CI low": np.percentile(diff, 2.5), "95% CI high": np.percentile(diff, 97.5),
                         "p (unadjusted)": boot_p(diff), "Δ BA": a["ba0"] - b_["ba0"]})
    return pd.DataFrame(rows)


def method_means_table(stat):
    """Method-level ΔEOdds with basic (reflected) and percentile intervals: a max - min gap is biased upward under
    resampling, so the basic interval [2θ - q97.5, 2θ - q2.5] is reported next to the percentile interval."""
    rows = []
    for n in CFG["confirmatory_budgets"]:
        for m in BOOT_COMPARE:
            st, th0 = stat[(m, n)], stat[(m, n)]["deo0"]
            q_lo, q_hi = np.percentile(st["deo"], [2.5, 97.5])
            rows.append({"n": n, "method": m, "mean ΔEOdds": th0,
                         "basic 95% CI": f"[{2 * th0 - q_hi:.3f}, {2 * th0 - q_lo:.3f}]",
                         "percentile 95% CI": f"[{q_lo:.3f}, {q_hi:.3f}]", "bootstrap bias": st["deo"].mean() - th0,
                         "mean BA": st["ba0"]})
    return pd.DataFrame(rows)


def per_pair_table(pair_boot):
    """Exploratory: per-pair BN-LFA ΔEOdds (seed-averaged) with household-bootstrap basic intervals."""
    n_big = max(CFG["confirmatory_budgets"])
    rows = []
    for (pair, n, m), (d0, db) in pair_boot.items():
        if m == "BN-LFA" and n == n_big:
            q_lo, q_hi = np.percentile(db, [2.5, 97.5])
            rows.append({"pair": pair, "BN-LFA ΔEOdds": d0, "basic 95% CI low": 2 * d0 - q_hi,
                         "basic 95% CI high": 2 * d0 - q_lo})
    return pd.DataFrame(rows)


def survey_weighted(doms, store):
    """Person-weighted (PWGTP) ΔEOdds and BA at n = max budget, seed 0, with replicate-weight standard errors, and
    the replicate SE of the paired BN-LFA − TT difference."""
    n_big = max(CFG["confirmatory_budgets"])
    sw_rows = []
    for s_tag, t_tag in CFG["pairs"]:
        te = doms[t_tag]["test"]
        repw = load_repweights(t_tag)[doms[t_tag]["idx_test"]]
        upos = np.bincount(te["G"][te["Y"] == 1].astype(int), minlength=N_GROUPS)
        uneg = np.bincount(te["G"][te["Y"] == 0].astype(int), minlength=N_GROUPS)
        support = (upos >= CFG["min_eval"]) & (uneg >= CFG["min_eval"])
        vals = {}
        for m in SW_METHODS:
            key = (s_tag, t_tag, n_big, 0, m)
            if key not in store:
                continue
            d = np.unpackbits(store[key], count=len(te["Y"])).astype(np.int8)

            def wmetrics(w):
                pos, neg, tp, fp = group_counts(te["Y"], d, te["G"], w)
                with np.errstate(divide="ignore", invalid="ignore"):
                    gm = _gaps(tp / pos, fp / neg, (tp + fp) / (pos + neg), support)
                return np.array([gm["d_eodds"], 0.5 * (tp.sum() / pos.sum() + 1 - fp.sum() / neg.sum())])

            full = wmetrics(te["PW"])
            reps = np.array([wmetrics(repw[:, r]) for r in range(80)])
            vals[m] = (full, reps)
            se = np.sqrt(4 / 80 * ((reps - full) ** 2).sum(0))
            sw_rows.append({"pair": f"{s_tag}→{t_tag}", "method": m, "weighted ΔEOdds": full[0], "SE ΔEOdds": se[0],
                            "weighted BA": full[1], "SE BA": se[1]})
        if "BN-LFA" in vals and "TT" in vals:
            dfull = vals["BN-LFA"][0] - vals["TT"][0]
            dse = np.sqrt(4 / 80 * (((vals["BN-LFA"][1] - vals["TT"][1]) - dfull) ** 2).sum(0))
            sw_rows.append({"pair": f"{s_tag}→{t_tag}", "method": "BN-LFA − TT", "weighted ΔEOdds": dfull[0],
                            "SE ΔEOdds": dse[0], "weighted BA": dfull[1], "SE BA": dse[1]})
    return pd.DataFrame(sw_rows)

### 1.12 Figures
Figures (each function takes the tables it plots, saves PNG + PDF to results/figures and returns the figure).

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8})

STYLES = {"EO-S": ("#7f7f7f", "--"), "TT": ("#1f77b4", "-"), "IW-EO": ("#9467bd", "-"), "LR-T": ("#8c564b", ":"),
          "GBT-T": ("#e377c2", ":"), "BN-S": ("#2ca02c", "--"), "BN-all": ("#bcbd22", "-"), "BN-LFA": ("#d62728", "-"),
          "BN-LL": ("#ff7f0e", "-."), "BN-rand": ("#17becf", "-."), "BN-LFA-dir": ("#aec7e8", ":"),
          "BN-LFA-emp": ("#98df8a", ":"), "BN-LFA-model": ("#c49c94", "--")}
CURVE_METHODS = ["EO-S", "TT", "IW-EO", "LR-T", "GBT-T", "BN-S", "BN-all", "BN-LFA"]
ABLATION_METHODS = ["EO-S", "BN-S", "BN-all", "BN-LFA", "BN-LL", "BN-rand", "BN-LFA-dir", "BN-LFA-model", "BN-LFA-emp"]


def draw_bn(ax, parents, title, highlight=(), domain_node=False):
    req_edges = required_edges()
    G = nx.DiGraph()
    G.add_nodes_from(VARS)
    for v in VARS:
        for p in parents[v]:
            G.add_edge(p, v)
    # staggered tier layout: no three nodes on one line, so edges do not run through other nodes
    pos = {"R": (0, 1.0), "X": (0, -1.0), "C": (1.1, 0.45), "E": (2.2, -0.55), "W": (3.3, 1.0), "O": (3.3, -1.0),
           "H": (4.4, 0.35), "Y": (5.6, -0.1)}
    colors = ["#d9822b" if v in highlight else ("#c9d6ea" if v in ("R", "X") else "#e8e8e8") for v in VARS]
    req = [(u, v) for u, v in G.edges if (u, v) in req_edges]
    learned = [(u, v) for u, v in G.edges if (u, v) not in req_edges and v != "Y"]
    to_y = [(u, v) for u, v in G.edges if v == "Y"]
    kw = {"ax": ax, "node_size": 900}
    nx.draw_networkx_edges(G, pos, edgelist=to_y, edge_color="#bdbdbd", arrowsize=8, width=0.7, style="dotted", **kw)
    nx.draw_networkx_edges(G, pos, edgelist=req, edge_color="#9db4d8", arrowsize=9, width=0.8,
                           connectionstyle="arc3,rad=0.08", **kw)
    nx.draw_networkx_edges(G, pos, edgelist=learned, edge_color="#222", arrowsize=13, width=1.8, **kw)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=900, edgecolors="#666", linewidths=0.6)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=10)
    if domain_node and highlight:
        pos_s = {**pos, "S": (2.5, 2.4)}
        H = nx.DiGraph()
        H.add_edges_from(("S", v) for v in highlight)
        nx.draw_networkx_nodes(H, pos_s, nodelist=["S"], ax=ax, node_color="#f2c14e", node_size=850)
        nx.draw_networkx_labels(H, pos_s, labels={"S": "S"}, ax=ax, font_size=9)
        nx.draw_networkx_edges(H, pos_s, ax=ax, style="dashed", edge_color="#b5651d", arrowsize=12)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def fig_learned_bns(structs, sources):
    fig, axes = plt.subplots(1, len(sources), figsize=(16, 4.2))
    for ax, tag in zip(np.atleast_1d(axes), sources):
        draw_bn(ax, structs[tag], f"{tag}: blue = required R,X edges, black = learned, dotted = outcome CPD")
    plt.tight_layout()
    save_fig(fig, "fig01_learned_bns")
    return fig


def fig_transfer_matrix(mats):
    """mats: {"BN-S": M, "EO-S": M}, M[source, target] = target ΔEOdds (diagonal: within-domain)."""
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
    labels = {"BN-S": "BN-S (model-based EO)", "EO-S": "EO-S (LR + EO post-proc.)"}
    for k, mt in enumerate(["BN-S", "EO-S"]):
        M, lab = mats[mt], labels[mt]
        deg = M.sub(np.diag(M.values), axis=0)
        for j, (mat, title, cmap) in enumerate([(M, f"target ΔEOdds: {lab}", "Reds"), (deg, f"degradation: {lab}", "RdBu_r")]):
            ax = axes[2 * k + j]
            vmax = np.nanmax(np.abs(mat.values))
            im = ax.imshow(mat.values.astype(float), cmap=cmap, vmin=(-vmax if cmap == "RdBu_r" else 0), vmax=vmax)
            ax.set_xticks(range(len(mat.columns)), mat.columns)
            ax.set_yticks(range(len(mat.index)), mat.index)
            ax.set_xlabel("target")
            ax.set_ylabel("source")
            ax.set_title(title, fontsize=9)
            for (i, jj), val in np.ndenumerate(mat.values.astype(float)):
                ax.text(jj, i, f"{val:.3f}", ha="center", va="center", fontsize=8)
            plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    save_fig(fig, "fig02_transfer_matrix")
    return fig


def budget_plot(summary, budgets, methods, fname, title, retrain=("LR-T", "GBT-T")):
    """ΔEOdds and BA against the label budget. Full-retraining baselines live on a different scale, so they get their
    own ΔEOdds panel (with TT repeated as a reference)."""
    budgets = sorted(budgets)
    xs = np.array([max(b, 50) for b in budgets])
    main = [m for m in methods if m not in retrain]
    rt = [m for m in methods if m in retrain]
    panels = [("d_eodds", main, "mean target ΔEOdds (lower is fairer)")]
    if rt:
        panels.append(("d_eodds", rt + ["TT"], "ΔEOdds: target-only retraining (TT for reference)"))
    panels.append(("ba", methods, "mean target balanced accuracy"))
    fig, axes = plt.subplots(1, len(panels), figsize=(5.4 * len(panels), 4))
    for ax, (met, ms, lab) in zip(axes, panels):
        for m in ms:
            sub = summary[summary.method == m].set_index("n").reindex(budgets)
            if sub[met].isna().all():
                continue
            col, ls = STYLES.get(m, ("k", "-"))
            ax.errorbar(xs, sub[met], yerr=sub[f"{met}_sd_seeds"].fillna(0), label=m, color=col, ls=ls, marker="o",
                        ms=3, capsize=2, lw=1.8 if m == "BN-LFA" else 1)
        ax.set_xscale("log")
        ax.set_xticks(xs, ["0" if b == 0 else str(b) for b in budgets])
        ax.minorticks_off()
        ax.set_xlabel("labelled target examples n")
        ax.set_ylabel(lab)
        ax.legend(ncol=2, frameon=False, fontsize=7)
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    save_fig(fig, fname)
    return fig


def fig_factor_shift(kl_df, fi_df, ui_df, expl):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), gridspec_kw={"width_ratios": [1, 1, 1, 0.9]})
    for ax, (df_, title, cmap) in zip(axes, [(kl_df, "KL(target‖source) per factor", "Purples"),
                                            (fi_df, "Δ ΔEOdds of source rule if only this factor shifts", "RdBu_r"),
                                            (ui_df, "Δ balanced accuracy if only this factor shifts", "RdBu_r")]):
        vals = df_.values.astype(float)
        vmax = np.nanmax(np.abs(vals)) or 1.0
        im = ax.imshow(vals, cmap=cmap, aspect="auto", vmin=(0 if cmap == "Purples" else -vmax), vmax=vmax)
        ax.set_xticks(range(len(VARS)), VARS)
        ax.set_yticks(range(len(df_)), df_.index, fontsize=8)
        ax.set_title(title, fontsize=9)
        for (i, j), val in np.ndenumerate(vals):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=6.5)
        plt.colorbar(im, ax=ax, fraction=0.04)
    ax = axes[3]
    ax.scatter(expl["predicted change"], expl["observed change (test)"], color="#d62728")
    for i, (_, r) in enumerate(expl.sort_values("predicted change").iterrows()):
        ax.annotate(r.pair, (r["predicted change"], r["observed change (test)"]), xytext=(4, 6 if i % 2 else -10),
                    textcoords="offset points", fontsize=6.5)
    lim = np.nanmax(np.abs(expl[["predicted change", "observed change (test)"]].values)) * 1.15
    ax.plot([-lim, lim], [-lim, lim], color="#999", lw=0.8, ls="--")
    ax.set_xlabel("model-predicted change in ΔEOdds")
    ax.set_ylabel("observed change on target test")
    ax.set_title("Does the factor shift explain the fairness change?", fontsize=9)
    plt.tight_layout()
    save_fig(fig, "fig05_factor_shift_diagnostic")
    return fig


def fig_decision_impact(diags, pair_names, n_big):
    dsub = diags[diags.n == n_big].groupby(["pair", "factor"])[["p_shift", "impact", "fair_impact"]].mean()
    shown = [p for p in pair_names[:2] + pair_names[6:7] if p in dsub.index.get_level_values(0)]
    fig, axes = plt.subplots(1, len(shown), figsize=(15, 3.2), sharey=True)
    for ax, pair in zip(np.atleast_1d(axes), shown):
        sub = dsub.loc[pair].reindex(VARS)
        ax.bar(VARS, sub["impact"], color=["#d62728" if p > 0.5 else "#bbbbbb" for p in sub["p_shift"]])
        ax.set_title(f"{pair}: expected decision impact (n = {n_big}, mean over seeds)", fontsize=8)
        ax.set_xlabel("factor (red: mean P(shift) > 0.5)")
    np.atleast_1d(axes)[0].set_ylabel("p_shift × (|ΔΔEOdds| + |ΔBA|)")
    plt.tight_layout()
    save_fig(fig, "fig06_decision_impact_examples")
    return fig


def fig_simulation(sim_main, sim_budgets, scenarios):
    fig, axes = plt.subplots(1, len(sim_budgets), figsize=(15, 3.8))
    plot_methods = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA", "BN-LFA +U"]
    for ax, n in zip(np.atleast_1d(axes), sim_budgets):
        sub = sim_main[sim_main.n == n].groupby(["scenario", "method"])["d_eodds"].agg(["mean", "std"]).reset_index()
        w = 0.11
        for k, m in enumerate(plot_methods):
            q = sub[sub.method == m].set_index("scenario").reindex(scenarios)
            col = STYLES.get(m.replace(" +U", ""), ("k", "-"))[0]
            ax.bar(np.arange(len(scenarios)) + (k - 3) * w, q["mean"], w, yerr=q["std"], label=m, color=col,
                   capsize=1.5, hatch="//" if m.endswith("+U") else None, edgecolor="white" if m.endswith("+U") else None)
        ax.set_xticks(range(len(scenarios)), scenarios)
        ax.set_ylabel("exact target ΔEOdds")
        ax.set_title(f"SYNTHETIC shift scenarios, n = {n}", fontsize=9)
    handles, labels_ = np.atleast_1d(axes)[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="lower center", ncol=len(plot_methods), frameon=False, fontsize=8)
    plt.tight_layout(rect=(0, 0.07, 1, 1))
    save_fig(fig, "fig07_simulation")
    return fig


def fig_tradeoff(to_mean, methods, n_big):
    """Seed-averaged trade-off curves: each point is one utility tolerance δ (circles: the preregistered δ)."""
    pairs = list(dict.fromkeys(to_mean.pair))
    fig, axes = plt.subplots(1, len(pairs), figsize=(12, 4))
    for ax, pair in zip(np.atleast_1d(axes), pairs):
        for m in methods:
            q = to_mean[(to_mean.pair == pair) & (to_mean.method == m)].sort_values("delta")
            col, ls = STYLES.get(m, ("k", "-"))
            ax.plot(q["ba"], q["d_eodds"], marker="o", ms=3, color=col, ls=ls, label=m)
            pref = q[np.isclose(q.delta, CFG["delta_ba"])]
            ax.scatter(pref["ba"], pref["d_eodds"], s=60, facecolors="none", edgecolors=col)
        ax.set_xlabel("target balanced accuracy")
        ax.set_ylabel("target ΔEOdds")
        ax.set_title(f"{pair}, n = {n_big}, mean of {len(CFG['tradeoff_seeds'])} seeds "
                     f"(circles: preregistered δ = {CFG['delta_ba']})", fontsize=9)
    np.atleast_1d(axes)[0].legend(frameon=False, fontsize=7)
    plt.tight_layout()
    save_fig(fig, "fig08_tradeoff_curves")
    return fig


def matrices_from_table(mat_out):
    """Inverse of the saved domain_transfer_matrix table."""
    return {m: g.drop(columns="method").set_index("source") for m, g in mat_out.groupby("method", sort=False)}

### 1.13 Pipeline stages
Each stage of the `Run` object writes its artifacts and returns its main tables.

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*divide by zero.*")
warnings.filterwarnings("ignore", message=".*non-interactive.*")

WITHIN_METHODS = {"LR-S": ("lr", "thr_lr_1"), "GBT-S": ("gbt", "thr_gbt_1"), "RW-S": ("rw", "thr_rw_1"),
                  "EO-S": ("lr", "thr_lr_eo"), "GBT-EO": ("gbt", "thr_gbt_eo"), "BN-S": ("bn", "thr_bn_eo")}
KEY = ["d_eodds", "d_eodds_30", "ba", "auroc", "ap", "d_eopp", "d_dp", "r_sel", "d_eodds_sex", "d_eodds_race",
       "logloss", "brier", "max_group_ece", "n_eval_groups"]
ABL = ["EO-S", "BN-S", "BN-LFA", "BN-LL", "BN-rand", "BN-all", "BN-LFA-dir", "BN-LFA-model", "BN-LFA-emp"]
ROB_METHODS = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA"]
ROB_SHOW = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA", "BN-LL", "BN-LFA (top excluded)"]
TO_METHODS = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA"]
PER_GROUP_METHODS = ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA"]
SIM_ALL = SIM_METHODS + [m + " +U" for m in SIM_U_METHODS]


def pair_names():
    return [f"{s}→{t}" for s, t in CFG["pairs"]]


def read_csv(path):
    """Read a saved table back exactly: only empty fields are missing (the scenario name "null" stays a string)."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""])


def stage(fn):
    """Record the running stage (for the artifact log) and its wall time."""
    @functools.wraps(fn)
    def wrapper(self, *a, **k):
        prev = CONTEXT.get("stage")
        CONTEXT["stage"] = fn.__name__
        t0 = time.time()
        try:
            return fn(self, *a, **k)
        finally:
            self.timings[fn.__name__] = self.timings.get(fn.__name__, 0.0) + time.time() - t0
            CONTEXT["stage"] = prev
    return wrapper


class Run:
    def __init__(self):
        self.raw = self.prov = self.doms = None
        self.card = card_of("primary")
        self.struct, self.bn_src, self.out_src, self.src = {}, {}, {}, {}
        self.mem, self.timings = {}, {}
        self.dec_store = None

    # ------------------------------------------------------------------------------------------ helpers
    def put(self, name, df, kind="tables"):
        self.mem[name] = df
        save_table(df, name, kind)
        return df

    def get(self, name, kind="tables"):
        if name not in self.mem:
            self.mem[name] = read_csv(PATHS[kind] / f"{name}.csv")
        return self.mem[name]

    def decisions(self):
        return self.dec_store

    def check(self, name, status):
        """Record a validation check in results/metrics/validation_checks.json."""
        path = PATHS["metrics"] / "validation_checks.json"
        checks = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        checks[name] = status
        save_json(checks, "validation_checks")

    def data(self):
        if self.doms is None:
            self.raw, self.prov = load_domains()
            self.doms = build_domains(self.raw, "primary")
        return self.doms


    # ------------------------------------------------------------------------------------------ data
    @stage
    def data_card(self):
        doms = self.data()
        prov_df = self.put("data_provenance", pd.DataFrame([self.prov[t] for t in CFG["domains"]]))
        status = {t: str(p.get("folktables_check")) for t, p in self.prov.items()}
        self.check("Folktables ACSIncome definition (rows, labels, features)", status)
        bad = [t for t, s in status.items() if not s.startswith(("passed", "skipped"))]
        assert not bad, f"ACSIncome verification failed for {bad}"
        card_df, comp, pos_rate = data_card_tables(doms)
        self.put("data_card_domains", card_df)
        self.put("data_card_group_counts", comp.reset_index().rename(columns={"index": "group"}))
        self.put("data_card_group_positive_rates", pos_rate.reset_index().rename(columns={"index": "group"}))
        return prov_df, card_df, (comp / comp.sum()).round(4), pos_rate

    # ------------------------------------------------------------------------------------------ graphical model
    @stage
    def structures(self):
        if self.struct:
            return self.mem.get("structure_learning"), self.mem.get("learned_edges_by_domain")
        doms, card = self.data(), self.card
        rows, hist, t0 = [], {}, time.time()
        for tag in CFG["domains"]:
            tr, va = doms[tag]["train"], doms[tag]["val"]
            self.struct[tag], hist[tag], score = learn_structure(tr, card, val=va)
            pa_bic = learn_structure(tr, card, score="bic")[0]
            self.bn_src[tag], self.out_src[tag] = fit_bn(tr, self.struct[tag], card)
            bn_bic, _ = fit_bn(tr, pa_bic, card, outcome=self.out_src[tag])
            npar = self.bn_src[tag].n_params()
            rows.append({"domain": tag,
                         "held-out structure": describe_structure(self.struct[tag]).replace("  P(Y|R,X,C,E,W,O,H)", ""),
                         "BIC structure": describe_structure(pa_bic).replace("  P(Y|R,X,C,E,W,O,H)", ""),
                         "val LL/person (held-out)": score / len(va["Y"]),
                         "val LL/person (BIC)": np.nansum([bn_bic.factor_loglik(va, v) for v in VARS if v != "Y"],
                                                          axis=0).mean(),
                         "composition params": sum(v for k, v in npar.items() if k != "Y"),
                         "outcome coefficients": self.out_src[tag].m.coef_.size + 1})
        print(f"structure learning + fitting: {time.time() - t0:.1f}s")
        struct_df = self.put("structure_learning", pd.DataFrame(rows))
        e = pd.DataFrame([{"domain": tag, "edge": f"{p}->{v}"} for tag in CFG["domains"] for v in COVS
                          for p in self.struct[tag][v] if p not in ("R", "X")], columns=["domain", "edge"])
        edge_tab = pd.crosstab(e["edge"], e["domain"]).reindex(columns=CFG["domains"]).fillna(0).astype(int)
        self.put("learned_edges_by_domain", edge_tab.reset_index())
        save_json({t: {v: list(p) for v, p in self.struct[t].items()} for t in CFG["domains"]}, "learned_structures")
        save_json({t: [list(h) for h in hist[t]] for t in CFG["domains"]}, "structure_search_history", kind="logs")
        first = CFG["domains"][0]
        print(f"Factorisation of the {first} network:\n  {describe_structure(self.struct[first])}")
        return struct_df, edge_tab

    @stage
    def inference_checks(self):
        doms = self.data()
        self.structures()
        tag, tol = CFG["inference_check_domain"], CFG["inference_tol"]
        col = f"pass (tol {tol:g})"
        rows = []
        for label, inc in (("full BN", True), ("composition sub-network", False)):
            res = run_inference_checks(self.bn_src[tag], inc, n_queries=CFG["inference_queries"],
                                       test=doms[tag]["test"])
            for k, v in res.items():
                passed = "n/a" if (isinstance(v, str) or k == "cliques in clique tree") else bool(v < tol)
                rows.append({"network": label, "check": k, "value": v, col: passed})
        check_df = self.put("inference_agreement_checks", pd.DataFrame(rows))
        ok = all(p in (True, "n/a") for p in check_df[col])
        worst = check_df[check_df[col] != "n/a"]["value"].astype(float).max()
        self.check("exact inference agreement (VE 4 orders = clique tree = enumeration = pgmpy)",
                   f"{'passed' if ok else 'FAILED'}: max abs diff {worst:.2e} (tol {tol:g}) on {tag}")
        assert ok, "exact-inference agreement check failed"
        cx_rows = []
        for t in CFG["sources"]:
            for label, inc, queries in (("composition", False, [("P(O | R=Black, X=F)", ["O"], {"R": 1, "X": 1}),
                                                              ("P(H | E=graduate)", ["H"], {"E": self.card["E"] - 1}),
                                                              ("P(R | O=service)", ["R"], {"O": 4})]),
                                        ("full", True, [("P(Y | R=Black, X=F)", ["Y"], {"R": 1, "X": 1}),
                                                        ("P(R | Y=1)", ["R"], {"Y": 1})])):
                fs = bn_factors(self.bn_src[t], inc)
                for qname, query, ev in queries:
                    for h in ["min_fill", "min_degree", "min_weight", "random"]:
                        st = {}
                        variable_elimination(fs, query, ev, h, rng=np.random.default_rng(1), stats=st)
                        cx_rows.append({"source": t, "network": label, "query": qname, "heuristic": h,
                                        "peak factor size": st["peak_factor"], "factor-entry ops": st["ops"],
                                        "order": "".join(st["order"])})
        cx_df = self.put("elimination_complexity", pd.DataFrame(cx_rows))
        return check_df, cx_df

    @stage
    def independence_checks(self):
        doms = self.data()
        self.structures()
        rows = []
        for tag in CFG["sources"]:
            pa, va = self.struct[tag], doms[tag]["val"]
            order_ = topological_order(pa)
            for v in COVS:
                for u in order_[:order_.index(v)]:
                    if u in pa[v]:
                        continue
                    g2, df, p, cmi, cmi_x = ci_test(va, v, u, pa[v], self.card)
                    rows.append({"source": tag, "statement": f"{v} ⊥ {u} | {{{','.join(pa[v])}}}",
                                 "d-separated": d_separated(pa, [v], [u], pa[v]), "G2": round(g2, 1), "df": df,
                                 "p": p, "CMI (bits)": round(cmi, 5), "bias-corrected CMI (bits)": round(cmi_x, 5)})
        ci_df = pd.DataFrame(rows, columns=["source", "statement", "d-separated", "G2", "df", "p", "CMI (bits)",
                                            "bias-corrected CMI (bits)"])
        assert ci_df["d-separated"].all(), "local Markov statements must be d-separations"
        self.check("local Markov statements of the learned graphs are d-separations", "passed")
        self.put("independence_checks", ci_df)
        print("R and X are parents of every covariate and of Y, so no conditional-independence statement removes a "
              "protected attribute from any factor.")
        return ci_df

    # ------------------------------------------------------------------------------------------ experiments
    @stage
    def fit_sources(self):
        if self.src:
            return self.src
        doms = self.data()
        self.structures()
        t0 = time.time()
        for tag in CFG["domains"]:
            d = doms[tag]
            self.src[tag] = Source(tag, d["train"], d["val"], self.card, bn=self.bn_src[tag], outcome=self.out_src[tag])
        print(f"fitted source models for {len(self.src)} domains in {time.time() - t0:.0f}s")
        return self.src

    @stage
    def within_domain(self):
        """Experiment 1: every rule evaluated on its own domain's test split, and the model-based audit."""
        doms = self.data()
        self.fit_sources()
        within_rows, audit_rows = [], []
        for tag in CFG["domains"]:
            S, te = self.src[tag], doms[tag]["test"]
            c = S.target_cache(tag, te)
            for m, (sk, tk) in WITHIN_METHODS.items():
                s = c[sk]
                within_rows.append({"domain": tag, "method": m,
                                    **evaluate(te["Y"], s, decide(s, te["G"], getattr(S, tk)), te["G"])})
            within_rows[-1]["bn_test_loglik_per_person"] = float(
                np.nansum([S.bn.factor_loglik(te, v) for v in VARS], axis=0).mean())
            # model-based audit accuracy: BN-implied vs empirical (test) group TPR/FPR of the EO-S rule
            _, _, tpr_m, fpr_m = table_metrics(model_tables(S.bn), S.thr_lr_eo, S.val_groups)
            pos, neg, tp, fp = group_counts(te["Y"], decide(c["lr"], te["G"], S.thr_lr_eo), te["G"])
            for g in range(N_GROUPS):
                if pos[g] >= CFG["min_eval_secondary"] and neg[g] >= CFG["min_eval_secondary"]:
                    audit_rows.append({"domain": tag, "group": GROUP_NAMES[g], "positives": int(pos[g]),
                                       "TPR model": tpr_m[g], "TPR test": tp[g] / pos[g], "FPR model": fpr_m[g],
                                       "FPR test": fp[g] / neg[g], "TPR 95% half-width": 1.96 * np.sqrt(
                                           max(tp[g] / pos[g] * (1 - tp[g] / pos[g]), 1e-9) / pos[g])})
        within = self.put("exp1_within_domain", pd.DataFrame(within_rows), "metrics")
        audit = pd.DataFrame(audit_rows)
        audit["|TPR error|"] = (audit["TPR model"] - audit["TPR test"]).abs()
        audit["|FPR error|"] = (audit["FPR model"] - audit["FPR test"]).abs()
        audit["within sampling noise"] = audit["|TPR error|"] <= audit["TPR 95% half-width"]
        self.put("model_based_audit_accuracy", audit, "metrics")
        self.check("fairlearn MetricFrame cross-check of per-group TPR/FPR", self._fairlearn_check())
        return within, audit

    def _fairlearn_check(self):
        try:
            from fairlearn.metrics import MetricFrame, true_positive_rate, false_positive_rate, selection_rate
            tag = CFG["inference_check_domain"]
            te = self.doms[tag]["test"]
            c = self.src[tag].target_cache(tag, te)
            d = decide(c["bn"], te["G"], self.src[tag].thr_bn_eo)
            mf = MetricFrame(metrics={"tpr": true_positive_rate, "fpr": false_positive_rate, "sel": selection_rate},
                             y_true=te["Y"], y_pred=d, sensitive_features=te["G"])
            pos, neg, tp, fp = group_counts(te["Y"], d, te["G"])
            present = np.where(pos + neg > 0)[0]
            ours = np.c_[tp[present] / np.maximum(pos[present], 1), fp[present] / np.maximum(neg[present], 1)]
            theirs = mf.by_group.loc[present, ["tpr", "fpr"]].to_numpy(float)
            ok = np.allclose(np.nan_to_num(ours), np.nan_to_num(theirs), atol=1e-12, equal_nan=True)
            res = "passed" if ok else "FAILED"
        except Exception as e:
            res = f"skipped ({type(e).__name__}: {e})"
        print(f"fairlearn MetricFrame cross-check of per-group TPR/FPR: {res}")
        return res

    @stage
    def group_calibration(self):
        """Per-group calibration of the source score models (LR-S, BN-S) within domain and after zero-label transfer."""
        doms = self.data()
        self.fit_sources()
        rows = []
        for s_tag, t_tag in [(t, t) for t in CFG["domains"]] + list(CFG["pairs"]):
            S, te = self.src[s_tag], doms[t_tag]["test"]
            c = S.target_cache(t_tag, te)
            for model, key in (("LR-S", "lr"), ("BN-S", "bn")):
                s = np.clip(c[key], 1e-6, 1 - 1e-6)
                ece = ece_by_group(te["Y"], s, te["G"], np.ones(N_GROUPS, bool))
                for g in range(N_GROUPS):
                    m = te["G"] == g
                    if not m.any():
                        continue
                    y, sg = te["Y"][m].astype(float), s[m]
                    pos, neg = int(y.sum()), int(len(y) - y.sum())
                    rows.append({"setting": "within-domain" if s_tag == t_tag else "zero-label transfer",
                                 "source": s_tag, "target": t_tag, "score model": model, "group": GROUP_NAMES[g],
                                 "test persons": int(m.sum()), "positives": pos, "negatives": neg,
                                 "support": support_label(pos, neg), "mean score": float(sg.mean()),
                                 "positive rate": float(y.mean()), "mean score - positive rate": float(sg.mean() - y.mean()),
                                 "ECE (10 bins)": float(ece[g]), "Brier": float(((sg - y) ** 2).mean()),
                                 "log loss": float(-(y * np.log(sg) + (1 - y) * np.log(1 - sg)).mean())})
        return self.put("group_calibration", pd.DataFrame(rows), "metrics")

    @stage
    def transfer(self):
        """Experiments 2, 3 and 5: every pair x budget x seed, all methods; decisions kept at the confirmatory budgets."""
        doms = self.data()
        self.fit_sources()
        RUNS, DIAGS, store = [], [], {}
        t0 = time.time()
        for s_tag, t_tag in CFG["pairs"]:
            tp = time.time()
            for n in CFG["budgets"]:
                for seed in (CFG["seeds"] if n in CFG["confirmatory_budgets"] else CFG["seeds_other"]) if n > 0 else [0]:
                    rows, diag = run_one(self.src[s_tag], t_tag, n, seed, doms,
                                         store=store if n in CFG["confirmatory_budgets"] else None)
                    RUNS.extend(rows)
                    if diag is not None:
                        DIAGS.append(diag.assign(source=s_tag, target=t_tag, pair=f"{s_tag}→{t_tag}", n=n, seed=seed))
            print(f"  {s_tag}→{t_tag}: {time.time() - tp:.0f}s")
        runs = self.put("exp_transfer_all_runs", pd.DataFrame(RUNS), "metrics")
        diags = self.put("factor_diagnostics_all_runs", pd.concat(DIAGS, ignore_index=True), "metrics")
        self.dec_store = store
        self.put("failed_configurations", pd.DataFrame(FAILED, columns=["source", "target", "n", "seed", "method",
                                                                        "reason"]), "logs")
        print(f"transfer grid: {len(runs):,} method-runs in {(time.time() - t0) / 60:.1f} min")
        return runs, diags

    @stage
    def explain_shift(self):
        """Factor-shift explanation on each target's full adaptation pool (P1 + P2, no test rows)."""
        doms = self.data()
        self.fit_sources()
        pool_rows, kl_tab, fi_tab, ui_tab = [], {}, {}, {}
        for s_tag, t_tag in CFG["pairs"]:
            S, T = self.src[s_tag], doms[t_tag]
            pool = take(T["all"], T["idx_pool"])
            common = np.intersect1d(support_groups(S.va["Y"], S.va["G"], CFG["min_eval"]),
                                    support_groups(pool["Y"], pool["G"], 4 * CFG["min_eval"]))
            dg, new = factor_diagnostic(S.bn, S.thr_lr_eo, pool, common)
            kls = {}
            for v in VARS:
                K = self.card[v]
                cnt = family_counts(pool, v, S.bn.parents[v], self.card).reshape(-1, K)
                w_u = cnt.sum(1) / cnt.sum()
                th_t, th_s = new[v].reshape(-1, K), S.bn.cpts[v].reshape(-1, K)
                kls[v] = float((w_u[:, None] * th_t * np.log(th_t / th_s)).sum())
            name = f"{s_tag}→{t_tag}"
            kl_tab[name] = kls
            fi_tab[name] = dict(zip(dg.factor, dg.fair_impact))
            ui_tab[name] = dict(zip(dg.factor, dg.util_impact))
            deo_src, ba_src, _, _ = table_metrics(model_tables(S.bn), S.thr_lr_eo, common)
            deo_all, ba_all, _, _ = table_metrics(model_tables(S.bn.copy(**new), S.bn), S.thr_lr_eo, common)
            te_s, te_t = doms[s_tag]["test"], T["test"]
            obs_s = gap_on(te_s["Y"], decide(S.target_cache(s_tag, te_s)["lr"], te_s["G"], S.thr_lr_eo), te_s["G"], common)
            obs_t = gap_on(te_t["Y"], decide(S.target_cache(t_tag, te_t)["lr"], te_t["G"], S.thr_lr_eo), te_t["G"], common)
            pool_rows.append({"pair": name, "groups": len(common), "model: source ΔEOdds": deo_src,
                              "model: all factors substituted": deo_all, "predicted change": deo_all - deo_src,
                              "observed: source test": obs_s[0], "observed: target test": obs_t[0],
                              "observed change (test)": obs_t[0] - obs_s[0], "predicted BA change": ba_all - ba_src,
                              "observed BA change (test)": obs_t[1] - obs_s[1],
                              "largest |single-factor impact|": dg.loc[dg.fair_impact.abs().idxmax(), "factor"],
                              "largest KL": max(kls, key=kls.get)})
        kl_df, fi_df, ui_df = (pd.DataFrame(x).T[VARS] for x in (kl_tab, fi_tab, ui_tab))
        expl = self.put("factor_shift_explanation", pd.DataFrame(pool_rows))
        for nm, df_ in (("kl", kl_df), ("fair_impact", fi_df), ("util_impact", ui_df)):
            self.put(f"factor_shift_full_pool_{nm}", df_.reset_index().rename(columns={"index": "pair"}))
        return expl

    @stage
    def simulation(self):
        doms = self.data()
        self.structures()
        sim, sim_diag = run_simulation(self.bn_src[CFG["sim_base_domain"]], dict(self.card),
                                       doms[CFG["sim_demographic_from"]]["train"])
        self.put("sim_all_runs", sim, "metrics")
        self.put("sim_diagnostic_recovery", sim_diag, "metrics")
        return sim, sim_diag

    @stage
    def robustness(self):
        """Feature bins, pooled two-state sources, forced exclusion of the top factor, unlabelled covariates."""
        doms = self.data()
        self.fit_sources()
        runs = self.get("exp_transfer_all_runs", "metrics")
        rob_rows, t0 = [], time.time()
        for scheme in CFG["robust_schemes"]:
            card_s = card_of(scheme)
            doms_s = build_domains(self.raw, scheme)
            for s_tag in CFG["sources"]:
                d = doms_s[s_tag]
                S = Source(f"{s_tag}[{scheme}]", d["train"], d["val"], card_s, gbt=False, rw=False)
                for t_tag in [t for s, t in CFG["pairs"] if s == s_tag]:
                    for n in [0] + CFG["robust_budgets"]:
                        for seed in (CFG["robust_seeds"] if n > 0 else [0]):
                            rows, _ = run_one(S, t_tag, n, seed, doms_s, methods=ROB_METHODS)
                            for r in rows:
                                r.update({"setting": f"bins: {scheme}", "pair": f"{s_tag}→{t_tag}"})
                            rob_rows.extend(rows)
            del doms_s
            print(f"  bins={scheme}: done ({time.time() - t0:.0f}s)")
        for a, b, t_tag in CFG["robust_pooled"]:
            tr = concat(doms[a]["train"], doms[b]["train"])
            va = concat(doms[a]["val"], doms[b]["val"])
            S = Source(f"{a}+{b}", tr, va, self.card, gbt=False, rw=False)
            for n in [0] + CFG["robust_budgets"]:
                for seed in (CFG["robust_seeds"] if n > 0 else [0]):
                    rows, _ = run_one(S, t_tag, n, seed, doms, methods=ROB_METHODS)
                    for r in rows:
                        r.update({"setting": "pooled two-state source", "pair": f"{a}+{b}→{t_tag}"})
                    rob_rows.extend(rows)
        print(f"  pooled sources: done ({time.time() - t0:.0f}s)")
        n_big = max(CFG["confirmatory_budgets"])
        for s_tag, t_tag in CFG["pairs"]:
            for seed in CFG["robust_seeds"]:
                sel = runs[(runs.pair == f"{s_tag}→{t_tag}") & (runs.n == n_big) & (runs.seed == seed)
                           & (runs.method == "BN-LFA")]
                top = str(sel["factors"].iloc[0]).split("+")[0] if len(sel) else "-"
                if top not in VARS:
                    continue
                rows, _ = run_one(self.src[s_tag], t_tag, n_big, seed, doms, methods=["BN-LFA"], exclude_factors=(top,))
                for r in rows:
                    r.update({"setting": "top factor excluded", "excluded": top, "method": "BN-LFA (top excluded)"})
                rob_rows.extend(rows)
        print(f"  forced exclusion: done ({time.time() - t0:.0f}s)")
        for s_tag, t_tag in CFG["pairs"]:
            for n in CFG["robust_budgets"]:
                for seed in CFG["robust_seeds"]:
                    rows, _ = run_one(self.src[s_tag], t_tag, n, seed, doms, unlabeled=CFG["unlabeled_pool"],
                                      methods=["IW-EO", "BN-all", "BN-LFA", "BN-LL"])
                    for r in rows:
                        r.update({"setting": "unlabelled covariates (+5k)"})
                    rob_rows.extend(rows)
        print(f"  unlabelled covariates: done ({time.time() - t0:.0f}s)")
        return self.put("robustness_all_runs", pd.DataFrame(rob_rows), "metrics")

    @stage
    def tradeoff(self):
        """Utility-fairness trade-off: re-run at n = max budget with utility tolerances δ (source rules re-selected)."""
        doms = self.data()
        self.fit_sources()
        n_big = max(CFG["confirmatory_budgets"])
        to_rows = []
        for s_tag, t_tag in CFG["tradeoff_pairs"]:
            for dlt in CFG["tradeoff_deltas"]:
                for seed in CFG["tradeoff_seeds"]:
                    rows, _ = run_one(self.src[s_tag], t_tag, n_big, seed, doms, methods=TO_METHODS, delta=dlt)
                    for r in rows:
                        r["delta"] = dlt
                    to_rows.extend(rows)
        return self.put("tradeoff_curves", pd.DataFrame(to_rows), "metrics")

    # ------------------------------------------------------------------------------------------ evaluation
    @stage
    def bootstrap(self):
        doms = self.data()
        t0 = time.time()
        stat, pair_boot = household_bootstrap(doms, self.decisions())
        print(f"bootstrap ({CFG['n_boot']} household replicates per target): {time.time() - t0:.0f}s")
        conf = self.put("confirmatory_tests_holm", confirmatory_table(stat), "metrics")
        expl = self.put("exploratory_tests_vs_no_adaptation", exploratory_table(stat), "metrics")
        self.put("bootstrap_method_means", method_means_table(stat), "metrics")
        self.put("bootstrap_per_pair_bnlfa", per_pair_table(pair_boot), "metrics")
        return conf, expl

    @stage
    def survey_weights(self):
        return self.put("survey_weighted_estimates", survey_weighted(self.data(), self.decisions()), "metrics")

    @stage
    def per_group(self):
        """Per-group denominators, selection rates, TPR and FPR of every main method at the confirmatory budgets."""
        doms = self.data()
        rows = []
        for (s_tag, t_tag, n, seed, m), bits in self.decisions().items():
            if m not in PER_GROUP_METHODS:
                continue
            te = doms[t_tag]["test"]
            d = np.unpackbits(bits, count=len(te["Y"])).astype(np.int8)
            for r in per_group_rates(te["Y"], d, te["G"]):
                rows.append({"pair": f"{s_tag}→{t_tag}", "source": s_tag, "target": t_tag, "n": n, "seed": seed,
                             "method": m, **r})
        return self.put("per_group_metrics", pd.DataFrame(rows), "metrics")

    # ------------------------------------------------------------------------------------------ results
    @stage
    def summaries(self):
        runs = self.get("exp_transfer_all_runs", "metrics")
        within = self.get("exp1_within_domain", "metrics")
        CB = CFG["confirmatory_budgets"]
        n_big = max(CB)
        # transfer summary (mean over pairs of the seed means; seed sd of the pair-averaged metric)
        per_pair = runs.groupby(["pair", "source", "target", "n", "method"], sort=False)[KEY].mean().reset_index()
        seed_level = runs.groupby(["n", "method", "seed"])[["d_eodds", "ba"]].mean()
        seed_sd = seed_level.groupby(["n", "method"]).std().add_suffix("_sd_seeds")
        summary = per_pair.groupby(["n", "method"])[KEY].mean()
        summary["worst_pair_d_eodds"] = per_pair.groupby(["n", "method"])["d_eodds"].max()
        summary["worst_target_ba"] = per_pair.groupby(["n", "method"])["ba"].min()
        summary = summary.join(seed_sd).reset_index()
        summary["family"] = summary["method"].map(METHOD_FAMILY)
        summary["protected attributes at prediction time"] = summary["method"].map(USES_PROTECTED)
        self.put("exp_transfer_summary", summary)
        self.put("exp_transfer_per_pair", per_pair)
        # domain-transfer matrix (zero labels) and year transfer
        first_year = str(CFG["years"][0])[2:]
        states = [d for d in CFG["domains"] if d.endswith(first_year)]

        def transfer_matrix(method):
            M = pd.DataFrame(index=states, columns=states, dtype=float)
            for s in states:
                for t in states:
                    if s == t:
                        M.loc[s, t] = within[(within.domain == s) & (within.method == method)]["d_eodds"].iloc[0]
                    else:
                        M.loc[s, t] = per_pair[(per_pair.source == s) & (per_pair.target == t) & (per_pair.n == 0)
                                               & (per_pair.method == method)]["d_eodds"].iloc[0]
            return M

        mats = {m: transfer_matrix(m) for m in ("BN-S", "EO-S")}
        self.put("domain_transfer_matrix", pd.concat(mats, names=["method", "source"]).reset_index())
        year_pairs = [f"{s}→{t}" for s, t in CFG["pairs"] if s[:2] == t[:2]]
        cols = ["BN-S", "EO-S", "LR-S", "GBT-S"]
        year = per_pair[(per_pair.n == 0) & per_pair.pair.isin(year_pairs) & per_pair.method.isin(cols)][
            ["pair", "method", "d_eodds", "ba"]]
        src_ref = within[within.method.isin(cols)][["domain", "method", "d_eodds", "ba"]]
        year = year.assign(source=year.pair.str[:4]).merge(src_ref, left_on=["source", "method"],
                                                           right_on=["domain", "method"], suffixes=("_2019", "_2018_within"))
        year["change_d_eodds"] = year["d_eodds_2019"] - year["d_eodds_2018_within"]
        self.put("year_transfer", year)
        # per-pair tables and ablations at the largest confirmatory budget
        order = [m for m in ALL_METHODS if m in summary["method"].unique()]
        pp = per_pair[per_pair.n == n_big]
        present = [m for m in order if m in pp.method.unique()]
        self.put(f"per_pair_d_eodds_n{n_big}", pp.pivot(index="pair", columns="method", values="d_eodds")[present].reset_index())
        self.put(f"per_pair_ba_n{n_big}", pp.pivot(index="pair", columns="method", values="ba")[present].reset_index())
        abl = summary[summary.method.isin(ABL)]
        abl_tab = abl.pivot(index="method", columns="n", values="d_eodds").reindex(ABL)
        self.put("ablation_d_eodds", abl_tab.reset_index())
        self.put("ablation_ba", abl.pivot(index="method", columns="n", values="ba").reindex(abl_tab.index).reset_index())
        # factor-selection frequency
        names = pair_names()

        def selection_frequency(method, n):
            sub = runs[(runs.method == method) & (runs.n == n)]
            tab = pd.DataFrame(0.0, index=names, columns=VARS)
            for _, r in sub.iterrows():
                for f in str(r.get("factors", "-")).split("+"):
                    if f in VARS:
                        tab.loc[r.pair, f] += 1
            return tab.div(sub.groupby("pair").size().reindex(names).to_numpy(), axis=0)

        for m in ("BN-LFA", "BN-LL"):
            for n in CB:
                self.put(f"selection_frequency_{m}_n{n}",
                         selection_frequency(m, n).reset_index().rename(columns={"index": "pair"}))
        # held-out factor log-likelihoods (probabilistic quality of the adapted networks)
        diags = self.get("factor_diagnostics_all_runs", "metrics")
        d = diags[diags.n.isin(CB)].assign(delta_ll=lambda x: x.test_ll_updated - x.test_ll_source,
                                           bf_positive=lambda x: x.log_bf > 0)
        fac = d.groupby(["n", "factor"])[["test_ll_source", "test_ll_updated", "delta_ll", "log_bf",
                                          "bf_positive"]].mean().reset_index()
        self.put("factor_heldout_loglik", fac.rename(columns={"bf_positive": "share of runs with log BF > 0"}))
        lfa = runs[(runs.method == "BN-LFA") & runs.n.isin(CB)].set_index(["pair", "n", "seed"])["factors"]
        jrows = []
        for (pair, n, seed), g in d.groupby(["pair", "n", "seed"], sort=False):
            F = [f for f in str(lfa.get((pair, n, seed), "-")).split("+") if f in VARS]
            base_ll = g.test_ll_source.sum()
            jrows.append({"pair": pair, "n": n, "seed": seed, "BN-LFA factors": "+".join(F) or "-",
                          "source BN": base_ll, "BN-LFA (selected factors updated)": base_ll + g[g.factor.isin(F)].delta_ll.sum(),
                          "BN-all (every factor updated)": g.test_ll_updated.sum()})
        joint = self.put("adapted_bn_heldout_loglik", pd.DataFrame(jrows), "metrics")
        self.put("adapted_bn_heldout_loglik_summary", joint.groupby("n")[
            ["source BN", "BN-LFA (selected factors updated)", "BN-all (every factor updated)"]].mean().reset_index())
        # simulation
        sim, sim_diag = self.get("sim_all_runs", "metrics"), self.get("sim_diagnostic_recovery", "metrics")
        sim_main = sim[sim.method.isin(SIM_ALL)]
        sim_tab = sim_main.groupby(["scenario", "n", "method"])[["d_eodds", "ba"]].agg(["mean", "std"]).round(3)
        rec = sim_diag.groupby(["scenario", "n", "selector"])[["recall", "false selections", "exact match"]].mean().round(2)
        est = sim[sim.method.str.startswith("estimate")]
        est_err = est.assign(abs_err=(est.d_eodds - est.true_source_rule_d_eodds).abs()).groupby(
            ["scenario", "n", "method"])["abs_err"].mean().unstack("method").reindex(SIM_SCENARIOS, level=0)
        self.put("sim_summary", sim_tab.reset_index())
        self.put("sim_recovery_summary", rec.reset_index())
        self.put("sim_estimation_error", est_err.reset_index())
        # robustness
        rob = self.get("robustness_all_runs", "metrics")
        base = runs[(runs.n.isin([0] + CFG["robust_budgets"])) & (runs.seed.isin(CFG["robust_seeds"] + [0]))].copy()
        base = base[(base.n > 0) | (base.seed == 0)]
        base["setting"] = "primary"
        comb = pd.concat([base, rob], ignore_index=True)
        rob_sum = comb[comb.method.isin(ROB_SHOW)].groupby(["setting", "n", "method"])[["d_eodds", "ba"]].mean()
        self.put("robustness_summary", rob_sum.reset_index())
        excl = rob[rob.setting == "top factor excluded"].merge(
            runs[(runs.method == "BN-LFA") & (runs.n == n_big)][["pair", "seed", "d_eodds", "ba", "factors"]],
            on=["pair", "seed"], suffixes=("_excluded", "_lfa"))
        if len(excl):
            excl["Δ ΔEOdds"] = excl["d_eodds_excluded"] - excl["d_eodds_lfa"]
            excl["Δ BA"] = excl["ba_excluded"] - excl["ba_lfa"]
            self.put("robustness_forced_exclusion", excl)
        # trade-off curves (seed means)
        to = self.get("tradeoff_curves", "metrics")
        self.put("tradeoff_curves_mean", to.groupby(["pair", "method", "delta"], sort=False)[
            ["d_eodds", "ba", "auroc"]].mean().reset_index())
        # per-group rates at the largest confirmatory budget (seed means; denominators are fixed by the test split)
        pg = self.get("per_group_metrics", "metrics")
        pgs = pg[pg.n == n_big].groupby(["pair", "method", "group"], sort=False).agg(
            persons=("persons", "first"), positives=("positives", "first"), negatives=("negatives", "first"),
            support=("support", "first"), selection_rate=("selection rate", "mean"), TPR=("TPR", "mean"),
            FPR=("FPR", "mean")).reset_index()
        self.put(f"per_group_summary_n{n_big}", pgs)
        return summary


    @stage
    def summary_markdown(self):
        within = self.get("exp1_within_domain", "metrics")
        audit = self.get("model_based_audit_accuracy", "metrics")
        per_pair, summary = self.get("exp_transfer_per_pair"), self.get("exp_transfer_summary")
        conf = self.get("confirmatory_tests_holm", "metrics")
        expl_cmp = self.get("exploratory_tests_vs_no_adaptation", "metrics")
        sim, sim_diag = self.get("sim_all_runs", "metrics"), self.get("sim_diagnostic_recovery", "metrics")
        expl = self.get("factor_shift_explanation")
        r_expl = stats.spearmanr(expl["predicted change"], expl["observed change (test)"])

        def f3(x):
            return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.3f}"

        def summ(n, m, col="d_eodds"):
            q = summary[(summary.n == n) & (summary.method == m)]
            return float(q[col].iloc[0]) if len(q) else np.nan

        nA, nB = CFG["confirmatory_budgets"]
        w_bn, w_eo = within[within.method == "BN-S"], within[within.method == "EO-S"]
        zero = per_pair[per_pair.n == 0].merge(within[["domain", "method", "d_eodds", "ba"]], left_on=["source", "method"],
                                               right_on=["domain", "method"], suffixes=("", "_src"))
        zero["deg"] = zero["d_eodds"] - zero["d_eodds_src"]
        lines = [f"**Run mode:** `{CFG['run_mode']}` · config `{CFG['config_file']}` · seeds {CFG['seeds']} · "
                 f"budgets {CFG['budgets']} · bootstrap B = {CFG['n_boot']}.", ""]
        lines.append("**Within-domain (Experiment 1).** Mean over the six domains: BN-S (model-based EO) ΔEOdds "
                     f"{f3(w_bn.d_eodds.mean())}, BA {f3(w_bn.ba.mean())}; EO-S (empirical EO, same scores) ΔEOdds "
                     f"{f3(w_eo.d_eodds.mean())}, BA {f3(w_eo.ba.mean())}; LR AUROC "
                     f"{f3(within[within.method == 'LR-S'].auroc.mean())}, GBT AUROC "
                     f"{f3(within[within.method == 'GBT-S'].auroc.mean())}. The model-implied group TPRs were within the "
                     f"95% sampling half-width of the test TPRs for {100 * audit['within sampling noise'].mean():.0f}% "
                     f"of group cells (mean |TPR error| {f3(audit['|TPR error|'].mean())}).")
        lines.append("")
        for m in ("EO-S", "BN-S"):
            z = zero[zero.method == m]
            lines.append(f"**Zero-label transfer, {m}.** Mean change in ΔEOdds relative to the source's own test split: "
                         f"{f3(z.deg.mean())} (range {f3(z.deg.min())} to {f3(z.deg.max())}); worst target ΔEOdds "
                         f"{f3(z.d_eodds.max())}.")
        lines.append("")
        lines.append(f"**Label-budgeted transfer (mean over {len(CFG['pairs'])} pairs).** n = {nA}: BN-LFA ΔEOdds "
                     f"{f3(summ(nA, 'BN-LFA'))} (BA {f3(summ(nA, 'BN-LFA', 'ba'))}), BN-all {f3(summ(nA, 'BN-all'))}, "
                     f"TT {f3(summ(nA, 'TT'))}, IW-EO {f3(summ(nA, 'IW-EO'))}, EO-S {f3(summ(nA, 'EO-S'))}, LR-T "
                     f"{f3(summ(nA, 'LR-T'))}. n = {nB}: BN-LFA {f3(summ(nB, 'BN-LFA'))} (BA {f3(summ(nB, 'BN-LFA', 'ba'))}), "
                     f"BN-all {f3(summ(nB, 'BN-all'))}, TT {f3(summ(nB, 'TT'))} (BA {f3(summ(nB, 'TT', 'ba'))}), "
                     f"IW-EO {f3(summ(nB, 'IW-EO'))}, LR-T {f3(summ(nB, 'LR-T'))}, GBT-T {f3(summ(nB, 'GBT-T'))}.")
        lines.append("")
        lines.append("**Confirmatory tests (household bootstrap, Holm).**")
        for _, r in conf.iterrows():
            verdict = ("supported" if r["fairness claim (Δ < 0, Holm α = 0.05)"] else
                       ("reversed (significantly worse)" if (r["p (Holm)"] < 0.05 and r["Δ mean ΔEOdds"] > 0)
                        else "not supported"))
            lines.append(f"* n = {r.n}, {r.comparison}: Δ = {f3(r['Δ mean ΔEOdds'])} [{f3(r['95% CI low'])}, "
                         f"{f3(r['95% CI high'])}], Holm p = {r['p (Holm)']:.3g} → **{verdict}**; ΔBA = {f3(r['Δ BA'])} "
                         f"(non-inferior at −0.02: {bool(r['BA non-inferior (CI low > −0.02)'])}).")
        lines.append("")
        lines.append("**Exploratory (unadjusted, not preregistered): adaptation vs. no adaptation.** " + "; ".join(
            f"n = {r.n}, {r['comparison (exploratory)']}: Δ = {f3(r['Δ mean ΔEOdds'])} [{f3(r['95% CI low'])}, "
            f"{f3(r['95% CI high'])}], p = {r['p (unadjusted)']:.3g}" for _, r in expl_cmp.iterrows()) + ".")
        lines.append("")
        abl_line = ", ".join(f"{m} {f3(summ(nB, m))}" for m in
                             ["EO-S", "BN-LFA", "BN-LL", "BN-rand", "BN-all", "BN-LFA-dir", "BN-LFA-model", "BN-LFA-emp"])
        lines.append(f"**Ablations (n = {nB}, mean ΔEOdds):** {abl_line}.")
        lines.append("")
        sm = sim[sim.method.isin(SIM_ALL)].groupby(["scenario", "method"])["d_eodds"].mean().unstack()
        sr = sim_diag[sim_diag.selector == "LFA"].groupby("scenario")[["recall", "exact match"]].mean()
        lines.append("**Simulation (synthetic).** Exact target ΔEOdds averaged over budgets — " + "; ".join(
            f"{sc}: EO-S {f3(sm.loc[sc, 'EO-S'])}, IW-EO {f3(sm.loc[sc, 'IW-EO'])}, TT {f3(sm.loc[sc, 'TT'])}, "
            f"BN-LFA {f3(sm.loc[sc, 'BN-LFA'])}" for sc in SIM_SCENARIOS if sc in sm.index) + ". LFA recall of the "
            "truly shifted factors: " + ", ".join(f"{sc} {f3(sr.loc[sc, 'recall'])}" for sc in SIM_SCENARIOS
                                                  if sc in sr.index and sc != "null") + ".")
        lines.append("")
        lines.append(f"**Factor-shift explanation.** Spearman correlation between model-predicted and observed change in "
                     f"ΔEOdds across the {len(expl)} pairs: {f3(r_expl.statistic)} (p = {r_expl.pvalue:.3f}; exploratory).")
        text = "\n".join(lines)
        save_text(text, "results_summary.md")
        print(text)
        return text

    def _data_version(self):
        hashes = []
        for t in CFG["domains"]:
            p = PATHS["processed"] / f"provenance_{t}.json"
            if p.exists():
                hashes.append(json.loads(p.read_text(encoding="utf-8")).get("zip_sha256", ""))
        label = "synthetic" if CFG.get("data_source") == "synthetic" else "ACS PUMS 1-Year person files"
        return f"{label}; sha256 of zip sha256s = {hashlib.sha256(''.join(hashes).encode()).hexdigest()[:16]}"

    @stage
    def reproducibility(self):
        sp, cp = PATHS["metrics"] / "learned_structures.json", PATHS["metrics"] / "validation_checks.json"
        repro = {"config": CFG, "versions": versions(), "data_version": self._data_version(),
                 "runtime_min": round((time.time() - T_START) / 60, 1), "data_provenance": self.prov,
                 "structures": json.loads(sp.read_text(encoding="utf-8")) if sp.exists() else {},
                 "validation_checks": json.loads(cp.read_text(encoding="utf-8")) if cp.exists() else {},
                 "failed_configurations": FAILED, "threshold_selection_events": dict(THRESHOLD_EVENTS)}
        save_json(repro, "reproducibility", kind="results")
        return repro

    @stage
    def manifest(self):
        """results/manifest.csv: every artifact with the stage that wrote it, settings hash, data version and SHA-256."""
        latest = {a["artifact"]: a for a in ARTIFACTS}
        rows = []
        for path, a in sorted(latest.items()):
            p = BASE_DIR / path
            rows.append({**a, "produced_by": f"{CFG['config_file']} (RUN_MODE={CFG['run_mode']})",
                         "config_sha256": CFG["config_sha256"], "seeds": f"split {CFG['split_seed']}; adaptation {CFG['seeds']}",
                         "data_version": self._data_version(), "status": "ok" if p.exists() else "missing",
                         "sha256": sha256_file(p) if p.exists() else "", "bytes": p.stat().st_size if p.exists() else 0})
        df = pd.DataFrame(rows)
        df.to_csv(PATHS["results"] / "manifest.csv", index=False)
        print(f"manifest: {len(df)} artifacts, {int((df.status == 'ok').sum())} ok")
        return df

## Part 2 · Experiments
### 2.1 Data acquisition, provenance and data card

In [ ]:
R = Run()
prov, card_df, comp_share, pos_rate = R.data_card()
display(prov[[c for c in ["domain", "rows_person_file", "rows_acsincome", "households_acsincome", "folktables_check",
                          "zip_sha256"] if c in prov.columns]])
display(card_df)
print("Group composition (share of persons) by domain:"); display(comp_share)
print("Positive rate P(Y = 1 | group) by domain:"); display(pos_rate)

### 2.2 Learned networks (structure identical in all six domains)

In [ ]:
struct_df, edge_tab = R.structures()
display(struct_df.round(4)); display(edge_tab)
fig_learned_bns(R.struct, CFG["sources"]);

### 2.3 Exact inference agreement and independence checks

In [ ]:
check_df, cx_df = R.inference_checks()
display(check_df); display(cx_df[cx_df.source == CFG["inference_check_domain"]])
display(R.independence_checks().sort_values(["source", "bias-corrected CMI (bits)"], ascending=[True, False]))

### 2.4 Experiment 1: within-domain performance, model-based audit, calibration by group

In [ ]:
within, audit = R.within_domain()
for col in ("d_eodds", "ba", "auroc"):
    print(col); display(within.pivot(index="method", columns="domain", values=col).round(3))
display(audit.groupby("domain")[["|TPR error|", "|FPR error|", "within sampling noise"]].mean().round(3))
calib = R.group_calibration()
display(calib[calib.support.str.startswith("primary")].groupby(["setting", "score model"])[
    ["ECE (10 bins)", "Brier", "log loss"]].mean().round(4))

### 2.5 Self-checks
Factor algebra, exact-inference agreement on a small random network, d-separation, metric definitions and edge cases,
Holm, household splits, and leakage: shuffling the target test labels must not change any decision.

In [ ]:
chk = np.random.default_rng(0)
f, g = Factor(["A", "B"], [2, 3], chk.random((2, 3))), Factor(["B", "C"], [3, 4], chk.random((3, 4)))
assert np.allclose(f.product(g).values, np.einsum("ab,bc->abc", f.values, g.values))
assert np.allclose(f.marginalize("A").values, f.values.sum(0)) and np.allclose(f.reduce({"B": 1}).values, f.values[:, 1])
small_card = {"R": 3, "X": 2, "C": 2, "E": 3, "W": 2, "O": 3, "H": 2, "Y": 2}
pa = R.struct["CA18"]
small = BayesNet(pa, small_card, {v: chk.dirichlet(np.ones(small_card[v]), size=int(np.prod([small_card[p] for p in pa[v]])))
                                  .reshape([small_card[p] for p in pa[v]] + [small_card[v]]) for v in VARS})
assert small.check() and abs(small.joint().sum() - 1) < 1e-12
res = run_inference_checks(small, True, n_queries=10, test=small.sample(100, chk))
assert all(v < 1e-9 for k, v in res.items() if k.startswith("max abs diff")), res
chain, collider = {"A": (), "B": ("A",), "C": ("B",)}, {"A": (), "B": (), "C": ("A", "B")}
assert d_separated(chain, ["A"], ["C"], ["B"]) and not d_separated(chain, ["A"], ["C"], [])
assert d_separated(collider, ["A"], ["B"], []) and not d_separated(collider, ["A"], ["B"], ["C"])
m = _gaps(np.array([0.9, 0.7]), np.array([0.1, 0.3]), np.array([0.5, 0.2]), [True, True])
assert np.isclose(m["d_eodds"], 0.2) and np.isclose(m["d_dp"], 0.3) and np.isclose(m["r_sel"], 0.4)
assert np.isnan(_gaps(np.zeros(2), np.zeros(2), np.zeros(2), [True, True])["r_sel"])      # undefined ratio
assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
for t, d in R.doms.items():                                                        # household-disjoint splits
    assert not set(d["all"]["HH"][d["idx_test"]]) & set(d["all"]["HH"][d["idx_pool"]])
leak_methods, a, b = ["EO-S", "TT", "IW-EO", "BN-all", "BN-LFA"], {}, {}
run_one(R.src["CA18"], "TX18", 200, 0, R.doms, methods=leak_methods, store=a)
shuffled, T = dict(R.doms), dict(R.doms["TX18"])
T["test"] = {**T["test"], "Y": chk.permutation(T["test"]["Y"])}
shuffled["TX18"] = T
run_one(R.src["CA18"], "TX18", 200, 0, shuffled, methods=leak_methods, store=b)
assert a.keys() == b.keys() and all(np.array_equal(a[k], b[k]) for k in a), "a decision depends on test labels"
R.check("self-checks (factors, inference, d-separation, metrics, Holm, splits, leakage)", "passed")
print("all self-checks passed")

### 2.6 Experiments 2, 3 and 5: transfer with fixed target-label budgets, and the factor-shift explanation
Every pair × budget × seed, all methods, the same labelled target persons for every method.

In [ ]:
runs, diags = R.transfer()
display(R.explain_shift().round(3))

### 2.7 Experiment 4: controlled simulation (SYNTHETIC: simulated numbers, not facts about the population)

In [ ]:
sim, sim_diag = R.simulation()

### 2.8 Robustness tests and trade-off curves

In [ ]:
rob = R.robustness()
to = R.tradeoff()

### 2.9 Paired household bootstrap with Holm correction, survey weights, per-group rates

In [ ]:
conf, expl_cmp = R.bootstrap()
display(conf.round(4)); display(expl_cmp.round(4))
sw = R.survey_weights()
display(sw.pivot(index="pair", columns="method", values="weighted ΔEOdds").round(3))
pg = R.per_group()

### 2.10 Summary tables and figures

In [ ]:
summary = R.summaries()
n_big = max(CFG["confirmatory_budgets"])
for n in CFG["confirmatory_budgets"]:
    print(f"Mean over the {len(CFG['pairs'])} pairs, n = {n}:")
    display(summary[summary.n == n].set_index("method")[["family", "d_eodds", "d_eodds_sd_seeds", "worst_pair_d_eodds",
            "d_eodds_30", "ba", "worst_target_ba", "auroc", "logloss", "brier", "d_dp", "d_eopp"]].round(3))
display(R.get("ablation_d_eodds").round(3))
display(R.get("robustness_summary").pivot_table(index=["setting", "n"], columns="method", values="d_eodds").round(3))
display(R.get("adapted_bn_heldout_loglik_summary").round(4))

In [ ]:
fig_transfer_matrix(matrices_from_table(R.get("domain_transfer_matrix")))
budgets = sorted(runs.n.unique())
budget_plot(summary, budgets, CURVE_METHODS, "fig03_budget_curves", f"Adaptation-budget curves ({len(CFG['pairs'])} pairs)")
budget_plot(summary, budgets, ABLATION_METHODS, "fig04_ablation_budget_curves", "Ablations of the proposed method")
kl, fi, ui = (R.get(f"factor_shift_full_pool_{k}").set_index("pair")[VARS] for k in ("kl", "fair_impact", "util_impact"))
fig_factor_shift(kl, fi, ui, R.get("factor_shift_explanation"))
fig_decision_impact(diags, pair_names(), n_big)
fig_simulation(sim[sim.method.isin(SIM_ALL)], CFG["sim_budgets"], SIM_SCENARIOS)
fig_tradeoff(R.get("tradeoff_curves_mean"), TO_METHODS, n_big);

### 2.11 Compact figures for the report and slides
Written to `results/figures/report/`; the report and slides use copies of these files.

In [ ]:
rep = PATHS["figures"] / "report"
rep.mkdir(exist_ok=True)
with plt.rc_context({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7}):
    fig, ax = plt.subplots(figsize=(5.6, 2.3))                                    # learned network
    draw_bn(ax, R.struct["CA18"], "")
    ax.set_xlim(-0.45, 6.05); ax.set_ylim(-1.4, 1.4)
    fig.savefig(rep / "bn_dag.pdf", bbox_inches="tight"); fig.savefig(rep / "bn_dag.png", dpi=220, bbox_inches="tight")
    xs = np.array([max(b, 50) for b in budgets])                                   # budget curves, main methods
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.4))
    for ax, col, lab in ((axes[0], "d_eodds", "mean target ΔEOdds (lower fairer)"), (axes[1], "ba", "mean target balanced accuracy")):
        for m in ["EO-S", "TT", "IW-EO", "BN-S", "BN-all", "BN-LFA"]:
            s = summary[summary.method == m].set_index("n").reindex(budgets)
            ax.errorbar(xs, s[col], yerr=s[f"{col}_sd_seeds"].fillna(0), color=STYLES[m][0], ls=STYLES[m][1], marker="o",
                        ms=2.5, capsize=1.5, lw=1.8 if m == "BN-LFA" else 1, label=m)
        ax.set_xscale("log"); ax.set_xticks(xs, ["0" if b == 0 else str(b) for b in budgets]); ax.minorticks_off()
        ax.set_xlabel("labelled target examples n"); ax.set_ylabel(lab)
    axes[0].legend(ncol=2, frameon=False); plt.tight_layout(); fig.savefig(rep / "budget_curves.pdf", bbox_inches="tight")
    fig, ax = plt.subplots(figsize=(4.2, 2.1))                                    # confirmatory paired differences
    y = np.arange(len(conf))[::-1]
    ax.errorbar(conf["Δ mean ΔEOdds"], y, xerr=[conf["Δ mean ΔEOdds"] - conf["95% CI low"],
                conf["95% CI high"] - conf["Δ mean ΔEOdds"]], fmt="o", color="#d62728", ms=3.5, capsize=2)
    ax.axvline(0, color="#888", lw=0.8, ls="--")
    ax.set_yticks(y, [f"vs {r['comparison'].split('− ')[-1]}, n = {r['n']}" for _, r in conf.iterrows()])
    for yy, (_, r) in zip(y, conf.iterrows()):
        ax.text(conf["95% CI high"].max() + 0.004, yy, f"Holm p = {r['p (Holm)']:.3f}", va="center", fontsize=6.5)
    ax.set_xlim(min(conf["95% CI low"].min() - 0.005, -0.08), conf["95% CI high"].max() + 0.04)
    ax.set_xlabel("BN-LFA minus comparator: mean target ΔEOdds (95% CI)"); plt.tight_layout()
    fig.savefig(rep / "confirmatory_forest.pdf", bbox_inches="tight")
    nsim, ms = max(CFG["sim_budgets"]), ["EO-S", "TT", "IW-EO", "BN-all", "BN-LFA", "BN-LFA +U"]   # simulation
    sub = sim[(sim.n == nsim) & sim.method.isin(ms)].groupby(["scenario", "method"])["d_eodds"].agg(["mean", "std"])
    fig, ax = plt.subplots(figsize=(7.0, 2.2))
    for k, m in enumerate(ms):
        q = sub.xs(m, level="method").reindex(SIM_SCENARIOS)
        ax.bar(np.arange(len(SIM_SCENARIOS)) + (k - 2.5) * 0.13, q["mean"], 0.13, yerr=q["std"], capsize=1, label=m,
               color=STYLES[m.replace(" +U", "")][0], hatch="//" if m.endswith("+U") else None,
               edgecolor="white" if m.endswith("+U") else None)
    ax.set_xticks(range(len(SIM_SCENARIOS)), SIM_SCENARIOS); ax.set_ylabel("exact target ΔEOdds")
    ax.set_title(f"SYNTHETIC scenarios (known target network), n = {nsim}"); ax.legend(ncol=6, frameon=False, loc="upper left", fontsize=6.5)
    plt.tight_layout(); fig.savefig(rep / "simulation.pdf", bbox_inches="tight")
plt.close("all")
print("report figures:", sorted(p.name for p in rep.iterdir()))

### 2.12 Demonstration (one exact query, one BN-LFA adaptation)
For the oral defence: trace the variable-elimination order, compare with the clique tree and enumeration, and explain
the factor diagnostic of one CA18 → TX18 run with 200 labelled TX18 persons.

In [ ]:
bn = R.bn_src["CA18"]
ev, st = {"R": 1, "X": 1, "E": R.card["E"] - 1}, {}                    # Black woman with a master's degree or higher
p_ve = variable_elimination(bn_factors(bn), ["Y"], ev, "min_fill", stats=st)
p_ct = CliqueTree(bn_factors(bn), elimination_order(bn_factors(bn), list(VARS), "min_fill")).calibrate(ev).marginal("Y")
p_en = enumeration_query(bn.joint(), "Y", ev)
print(f"P(Y=1 | Black, female, master's+): VE {p_ve[1]:.6f} (order {''.join(st['order'])}, peak factor {st['peak_factor']}), "
      f"clique tree {p_ct[1]:.6f}, enumeration {p_en[1]:.6f}")
rows, diag = run_one(R.src["CA18"], "TX18", 200, 0, R.doms, methods=["EO-S", "TT", "IW-EO", "BN-all", "BN-LFA"])
display(diag[["factor", "family", "log_bf", "p_shift", "fair_impact", "util_impact", "impact", "update"]].round(4))
display(pd.DataFrame(rows).set_index("method")[["factors", "d_eodds", "ba", "auroc"]].round(3))

### 2.13 Results summary, reproducibility record and manifest

In [ ]:
display(Markdown(R.summary_markdown()))
repro = R.reproducibility()
manifest = R.manifest()
display(pd.DataFrame({"package": list(repro["versions"]), "version": list(repro["versions"].values())}))

In [ ]:
import shutil
archive = shutil.make_archive(str(BASE_DIR / "p5_results"), "zip", PATHS["results"])
print("results archive:", archive)
if IN_COLAB:
    try:
        from google.colab import files as colab_files
        colab_files.download(archive)
    except Exception as e:
        print(f"automatic download unavailable ({e}); use the Files panel to download {archive}")